# NTL Exp1 Cached Hybrid Soft MRF

Eksperimen **hybrid_mrf** dibandingkan dengan: cached-hybrid-v2; uncached hybrid v3 scored 0.602.

**Perubahan:** Replace only hard Hungarian with guarded Soft MRF; same hybrid mixtures and ATLAS routing.

Bobot LLM, adapter LoRA, readout dan Gaussian berasal dari input pretrained yang sama.
Tidak melatih ulang Qwen, Llama atau LoRA. Sepuluh cache fitur train dan tujuh Gaussian
tetap terikat ke kontrak sumber, official train serta checkpoint yang telah diverifikasi.
Test tersembunyi selalu dihitung dengan model; cache tiga contoh publik hanya berlaku
jika seluruh ID dan SHA teks lengkap cocok. Uji publik tidak membuktikan akurasi hidden.

Campuran sebelum ATLAS: baseline 0.75, Dhruv 0.20,
SVC gabungan 0.05. ATLAS: `hybrid-selective`.
Soft MRF memakai penalti **0.4 x confidence kelompok**;
seluruh batas rank, perubahan probabilitas, ukuran kelompok dan gap tetap sama dengan versi 0,608.
Semua decoder mengeluarkan label terpisah dan menjaga matriks probabilitas.

Varian 2/3/4 memakai panggilan ATLAS asli `predict(..., batch_size=1)` untuk seluruh baris.
Varian 1 mempertahankan routing ATLAS selektif, bobot dan batas waktu dari hybrid cache.
Cache fitur train dapat menimbulkan perbedaan numerik FP16 kecil akibat susunan batch,
sehingga kesamaan skor dengan eksekusi tanpa cache belum dijamin.

Untuk varian 3, temperatur baru berlaku hanya pada SVC tambahan. Tiga fold dikelompokkan
berdasarkan prompt ID/key dan alias yang terhubung melalui teks identik yang dinormalisasi.
NLL untuk memilih temperatur memakai baris official train yang diprediksi fold lain.
NLL sesudah fitting temperatur bukan estimasi independen akurasi ensemble. Duplikat prompt
semantik yang tidak diketahui masih mungkin tersisa; transfer temperatur dari estimator fold
ke SVC penuh adalah asumsi. Tidak ada fitting temperatur dari prediksi in-sample Qwen.

Semua berjalan pada **T4 x2**, Internet OFF, batas sesi 9 jam. Pemeriksaan menolak model
tanpa bobot, label/urutan ID berbeda, atau cache tidak cocok. Submission: `submission.parquet`.
Audit: `hybrid-report.json`, `hybrid-probabilities.npz`, `mrf-changes.csv`, `mrf-groups.csv`.
Varian 3 juga menyimpan `svc-calibration-report.json` dan skor OOF official train.

Referensi dan kredit: [baseline 0,606](https://www.kaggle.com/code/taqiyudinadn/ntl-public-atlas-reproduction?scriptVersionId=360593727),
[Soft MRF 0,608](https://www.kaggle.com/code/adityaakbar/ntl-atlas-soft-mrf-constraints?scriptVersionId=361000837),
[Dhruv](https://www.kaggle.com/code/dhruvtibarewal/nlm-v21),
[Vishak](https://www.kaggle.com/code/vishakkbhat/v1-vishak),
[ATLAS](https://www.kaggle.com/code/josephadamski91/teach-the-model-10-atlas),
[temperature scaling, Guo et al. 2017](https://proceedings.mlr.press/v70/guo17a.html),
[StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html).
Kredit aset tambahan tetap mengikuti dataset/kernel input dari baseline dan kontrak cache.


In [ ]:
import hashlib as _ntl_hashlib
import importlib.metadata as _ntl_packages
import json as _ntl_json
import os as _ntl_os
from pathlib import Path as _ntl_Path
import time as _ntl_time
import torch as _ntl_torch

_ntl_started = _ntl_time.time()
_ntl_os.environ["HF_HUB_OFFLINE"] = "1"
_ntl_os.environ["TRANSFORMERS_OFFLINE"] = "1"
if not _ntl_torch.cuda.is_available():
    raise RuntimeError("Enable T4 x2 before running the public ensemble")
_ntl_versions = {name: _ntl_packages.version(name) for name in
                 ("torch", "transformers", "peft", "accelerate", "safetensors",
                  "numpy", "pandas", "scipy", "scikit-learn", "lightgbm", "pyarrow")}
print("Public ensemble environment:", _ntl_versions, flush=True)
print("GPUs:", [_ntl_torch.cuda.get_device_name(i)
                for i in range(_ntl_torch.cuda.device_count())], flush=True)

def _ntl_file_sha256(path):
    digest = _ntl_hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

# Metadata and adapter fingerprints identify the assets actually resolved by
# Kaggle. Full multi-GB base weight files are excluded from this startup scan.
_ntl_inputs = {}
for _ntl_path in sorted(_ntl_Path("/kaggle/input").rglob("*")):
    if _ntl_path.is_file() and (_ntl_path.suffix == ".json" or
                              _ntl_path.name in ("heads.safetensors", "heads.pt", "norms.safetensors",
                                                 "adapter_model.safetensors", "adapters.safetensors")):
        _ntl_inputs[str(_ntl_path)] = {"bytes": _ntl_path.stat().st_size,
                                       "sha256": _ntl_file_sha256(_ntl_path)}
_ntl_Path("input-fingerprints.json").write_text(_ntl_json.dumps(_ntl_inputs, indent=2) + "\n")
print("Fingerprinted", len(_ntl_inputs), "input metadata/adapter files", flush=True)

CHECKPOINT_IDENTITY = {'pipeline': 'public-atlas-ensemble', 'upstream_code_sha256': 'bbc0cf75abc974664beb4fc8a6f2c3bed7c3f10156c62e6f1620db67f3ffff8b', 'atlas_adapter_sha256': '29e9b32e09679ae4e3b2cdf0729c7011ecee730fdb0f20982593af0b40806106', 'labels': ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']}
NOTEBOOK_BUILD_SHA256 = '8724de3ff2738e2e7bc54295b822129a51227d8e87d9c4c213abb00fbfbf6a39'


In [ ]:
_COPY_EXPECTED_MIRROR = {'name-that-llm-train-lora24-s7/lora_out/job_0/COMPLETE.json': {'bytes': 100, 'sha256': 'f404e4e588d42a5c0edb0e9d3502ca4f8f35d4b1db8aaf4c4132468c5f26d525'}, 'name-that-llm-train-lora24-s7/lora_out/job_1/COMPLETE.json': {'bytes': 101, 'sha256': '4af1d476bea5bc68229b83c9aae0a1f4a05217803beb39086341aff19e0b34e8'}, 'name-that-llm-train-lora24-s7/lora_out/job_2/COMPLETE.json': {'bytes': 101, 'sha256': '268e285d6c84b85cde42f8ff103c4b1a3e3f5bea7e64a29b86e4e8bb281607e8'}, 'name-that-llm-train-lora24-s7/lora_out/job_3/COMPLETE.json': {'bytes': 86, 'sha256': '6fae380abb1b61fd15ccfc3ecc2d84f07b6516008ff01c1e46cfdcd0ef283a9d'}, 'name-that-llm-train-lora24-s7/lora_out/job_4/COMPLETE.json': {'bytes': 101, 'sha256': '2dd1378a2d78f963145446fb6499a97292fed4b8d0ea3e42957c99572bc2bfaf'}, 'name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json': {'bytes': 104, 'sha256': 'e92a525842d44c1d4a7f3c55bbcdfdc46ed96a545b90b80717fc86f6cb22c2d5'}, 'name-that-llm-train-lora24-s7/lora_out/job_final/adapter/adapter_config.json': {'bytes': 1288, 'sha256': '3b786d83737e6890ccb19a2cddbaee5c8b1aae1bc2c13840161970909165711c'}, 'name-that-llm-train-lora24-s7/lora_out/job_final/adapter/adapter_model.safetensors': {'bytes': 116434760, 'sha256': 'a045cbdc521b957736e828a3d8f1eee4a844e5ce1a8c2a17e1b0531f86212682'}, 'name-that-llm-train-lora24-s7/lora_out/job_final/norms.safetensors': {'bytes': 263384, 'sha256': '709a8214889291cc9d779928bbdf3ba4849d97e023c005f6ba11f9b704d75126'}, 'name-that-llm-train-lora24-s7/lora_out/job_final/tokenizer/tokenizer.json': {'bytes': 11422748, 'sha256': 'b3744d3112b8c3d04760859bfdbf9a7ce00d5539512b5fc5d5bfb3f76eec0d9d'}, 'name-that-llm-train-lora24-s7/lora_out/job_final/tokenizer/tokenizer_config.json': {'bytes': 693, 'sha256': '2e3bd3cb5055f7a903a54e7062b2fd14d21b70f4174c3ffcd9fc81ec50a25631'}, 'ntl-lora24-fivefold-training/lora24/final/COMPLETE.json': {'bytes': 1310, 'sha256': '3a7706a211a393fd175aba2ed56ed03481c6f6e73904b5daf1bc40881c5680eb'}, 'ntl-lora24-fivefold-training/lora24/final/adapter/adapter_config.json': {'bytes': 1288, 'sha256': '255d5014cbb6981357f42eba6be2742ba90909bec14c10a3409178a85bc19085'}, 'ntl-lora24-fivefold-training/lora24/final/adapter/adapter_model.safetensors': {'bytes': 116434760, 'sha256': 'a6aff409ae2df897717875373d038635fe5012fbf93557c90aea3457fda0e7f8'}, 'ntl-lora24-fivefold-training/lora24/final/base_config/config.json': {'bytes': 1332, 'sha256': 'd9a04fdd6dd68c2129a07b40c2534823f77b7970192934759e35402e5cca1a3e'}, 'ntl-lora24-fivefold-training/lora24/final/heads.safetensors': {'bytes': 4982480, 'sha256': '2176ade29e6eb97bd3a6c204f944959f9757a80c6f1ff475898c23bce60ecd96'}, 'ntl-lora24-fivefold-training/lora24/final/tokenizer/tokenizer.json': {'bytes': 11422748, 'sha256': 'b3744d3112b8c3d04760859bfdbf9a7ce00d5539512b5fc5d5bfb3f76eec0d9d'}, 'ntl-lora24-fivefold-training/lora24/final/tokenizer/tokenizer_config.json': {'bytes': 692, 'sha256': '1cc816812993bff176eb4f7495433b736f06fba9b6e7b05cac7b4a1780650c95'}, 'ntl-lora24-fivefold-training/lora24/final/training_report.json': {'bytes': 542527, 'sha256': '98213860e48853812d5a3337b546309f183a674ab7bde2391e68d791e6dba8dc'}, 'ntl-lora24-fivefold-training/lora24/outer0/COMPLETE.json': {'bytes': 805, 'sha256': 'e184fdde02fd5d1ae1dca1bbee280c29cffbea3831e198d4da36e173ba99805f'}, 'ntl-lora24-fivefold-training/lora24/outer0/adapter/adapter_config.json': {'bytes': 1288, 'sha256': 'f469f6e589d89791547e8fde6bbcae8e622a9559d46b21ed137391b6ac36b300'}, 'ntl-lora24-fivefold-training/lora24/outer0/adapter/adapter_model.safetensors': {'bytes': 116434760, 'sha256': '108bcf16907a03f7c514805214fee8d363f4f0083365b12d9c1841d502185dd2'}, 'ntl-lora24-fivefold-training/lora24/outer0/heads.safetensors': {'bytes': 4982480, 'sha256': 'caf3a1f678c2015512170154e72fb24a4bac596fd7d3445131b476eeedef99a7'}, 'ntl-lora24-fivefold-training/lora24/outer0/training_report.json': {'bytes': 434605, 'sha256': 'c9fde672ced14731729eaf31a505bf6e5ce8a9daa220eb7209ac44cd07f7c10e'}, 'ntl-lora24-fivefold-training/lora24/outer1/COMPLETE.json': {'bytes': 805, 'sha256': 'cae6fd637ee283404aa9f1b6218da04d39247925686df48606dd57bfa1fc15ee'}, 'ntl-lora24-fivefold-training/lora24/outer1/adapter/adapter_config.json': {'bytes': 1288, 'sha256': '255d5014cbb6981357f42eba6be2742ba90909bec14c10a3409178a85bc19085'}, 'ntl-lora24-fivefold-training/lora24/outer1/adapter/adapter_model.safetensors': {'bytes': 116434760, 'sha256': '38a8ad13eef75e615c41ea327cf6af799a3a31b57a0f6071f5d2f3468d9d19f8'}, 'ntl-lora24-fivefold-training/lora24/outer1/heads.safetensors': {'bytes': 4982480, 'sha256': 'dcdf7628509c7cbd7fe0ef2b8920bb2384f2cea423f62a8b728de9b5d0411a8a'}, 'ntl-lora24-fivefold-training/lora24/outer1/training_report.json': {'bytes': 434155, 'sha256': '7dd951ee7bf8222c2e43babc15e342a2c58dc787f99e3022d7bcfddb7b3b5c43'}, 'ntl-lora24-fivefold-training/lora24/outer2/COMPLETE.json': {'bytes': 805, 'sha256': '1ba7da1ab694c06fd9f68dc1b9ba6da161cddbc9284705b104d248514a07ca50'}, 'ntl-lora24-fivefold-training/lora24/outer2/adapter/adapter_config.json': {'bytes': 1288, 'sha256': 'f469f6e589d89791547e8fde6bbcae8e622a9559d46b21ed137391b6ac36b300'}, 'ntl-lora24-fivefold-training/lora24/outer2/adapter/adapter_model.safetensors': {'bytes': 116434760, 'sha256': '107ea25c2df60d1556a93e4e0efbe9a236b0baa4cd1def8b6febc4f2fb1d372e'}, 'ntl-lora24-fivefold-training/lora24/outer2/heads.safetensors': {'bytes': 4982480, 'sha256': '861fa876d85535c6d5a0184f04de773b9096ede016c606eeaf94e102556d2bb5'}, 'ntl-lora24-fivefold-training/lora24/outer2/training_report.json': {'bytes': 434628, 'sha256': 'cfbfe5396ece892ae3dc758ebad79f7209fef5b73f40ec35752692e1925a6f47'}, 'ntl-lora24-fivefold-training/lora24/outer3/COMPLETE.json': {'bytes': 805, 'sha256': '3fd9fd3c4f6ee1bf6b7bb36594244203fb580644a95e4d5476805ffbde8139e6'}, 'ntl-lora24-fivefold-training/lora24/outer3/adapter/adapter_config.json': {'bytes': 1288, 'sha256': '255d5014cbb6981357f42eba6be2742ba90909bec14c10a3409178a85bc19085'}, 'ntl-lora24-fivefold-training/lora24/outer3/adapter/adapter_model.safetensors': {'bytes': 116434760, 'sha256': '8c3119e62a4fd0eb9366063cf3d9ccb8a9fdc6805143840552d257b0569ebee7'}, 'ntl-lora24-fivefold-training/lora24/outer3/heads.safetensors': {'bytes': 4982480, 'sha256': 'ed1adfcf24a2e80f4940b95b7fc5e9ea57064ca3f18ac669b8876cc2bde20ac3'}, 'ntl-lora24-fivefold-training/lora24/outer3/training_report.json': {'bytes': 434242, 'sha256': 'd7faaf2f0739153d09b21e1b7a32c99eb98aadc415d46b84439973427e480718'}, 'ntl-lora24-fivefold-training/lora24/outer4/COMPLETE.json': {'bytes': 805, 'sha256': '6d74dc5f9bb648258de76f6dbd820ec42436dfdcc9f2186090c80b4f7db92a05'}, 'ntl-lora24-fivefold-training/lora24/outer4/adapter/adapter_config.json': {'bytes': 1288, 'sha256': 'f469f6e589d89791547e8fde6bbcae8e622a9559d46b21ed137391b6ac36b300'}, 'ntl-lora24-fivefold-training/lora24/outer4/adapter/adapter_model.safetensors': {'bytes': 116434760, 'sha256': '0387a4d6d258fd739f4f57b0ebcef958af55aff3525ee17c413036064588b665'}, 'ntl-lora24-fivefold-training/lora24/outer4/heads.safetensors': {'bytes': 4982480, 'sha256': '665e61add4fed6efee7764af39514e277064eaa976053cc5585499eb5e2ca8f2'}, 'ntl-lora24-fivefold-training/lora24/outer4/training_report.json': {'bytes': 434668, 'sha256': '13059bfc14aea95ae10f24c4c7f5302335f088c190193d31b472903b6d7f2c0e'}, 'ntl-lora24-fivefold-training/lora24/worker0_status.json': {'bytes': 416, 'sha256': '2f482053a27eb2287e8217d3773d5504e4c9a6c668523e8d781a3cb49290214a'}, 'ntl-lora24-fivefold-training/lora24/worker1_status.json': {'bytes': 413, 'sha256': 'b50ae7b5c4b64fbe68914863b7ffcf6e8a3d8f7164c7b2127238d4d4f09989a5'}, 'ntl-lora24-fivefold-training/lora24_comparison.json': {'bytes': 2441, 'sha256': '55b05bd4e536df60d93605b19766b8653cc29596c85cdc081124ba402003b13a'}, 'ntl-lora24-fivefold-training/optional_torchao_audit.json': {'bytes': 909, 'sha256': 'a7eb23fae319bb71517016eeb99d8a811d44cba8198a84154c853133ed871f99'}, 'ntl-lora24-fivefold-training/source_audit.json': {'bytes': 363, 'sha256': '04a17ac576f425d189f145a09277542f12eaac7872fdf1c3c8a0d428829c03eb'}, 'ntl-lora24-fivefold-training/training_spec.json': {'bytes': 763843, 'sha256': '5edb565a30b8da089f2611929b4114f7bef589dd01ca528e976a3685caea4444'}, 'ntl-qwen3-8b-lora-claude-head/final/COMPLETE.json': {'bytes': 44, 'sha256': '4a9f1ee827240c221c2e83bc0686df644aa3d16c2cef08e0b5e1e71c463b2a54'}, 'ntl-qwen3-8b-lora-claude-head/final/adapter/adapter_config.json': {'bytes': 1288, 'sha256': 'e37c54313ad1d0886f0f4562e82e7934cea51b9433f7465fca9df9f765fe6176'}, 'ntl-qwen3-8b-lora-claude-head/final/adapter/adapter_model.safetensors': {'bytes': 116434760, 'sha256': '9aed9517bf5954c90c14ad22a95af64b90c3ad23407de2910c05bc18c804c505'}, 'ntl-qwen3-8b-lora-claude-head/final/heads.safetensors': {'bytes': 263384, 'sha256': '79fcbbc4b313450b4580565d5addba4cd769e80f10d5d68a2068e694b50619aa'}, 'ntl-qwen3-8b-lora-claude-head/final/tokenizer/tokenizer.json': {'bytes': 11422748, 'sha256': 'b3744d3112b8c3d04760859bfdbf9a7ce00d5539512b5fc5d5bfb3f76eec0d9d'}, 'ntl-qwen3-8b-lora-claude-head/final/tokenizer/tokenizer_config.json': {'bytes': 693, 'sha256': '2e3bd3cb5055f7a903a54e7062b2fd14d21b70f4174c3ffcd9fc81ec50a25631'}, 'ntl-qwen3-8b-lora-claude-head/fold0/COMPLETE.json': {'bytes': 44, 'sha256': '4021a8cd415dcae0a29535745def682ee04c7b65a414f06b8feafa343e8e9430'}, 'ntl-qwen3-8b-lora-claude-head/fold0/adapter/adapter_config.json': {'bytes': 1288, 'sha256': 'e37c54313ad1d0886f0f4562e82e7934cea51b9433f7465fca9df9f765fe6176'}, 'ntl-qwen3-8b-lora-claude-head/fold0/adapter/adapter_model.safetensors': {'bytes': 116434760, 'sha256': '0f6687e0dc56faac19e32a900915963f4df3b4f1c033a8c20fe8a15f3df3f196'}, 'ntl-qwen3-8b-lora-claude-head/fold0/heads.safetensors': {'bytes': 263384, 'sha256': 'a0bcb7c4b7e85d02e558800d30ea591778b28b0466b0fbd5c8208dc64c665289'}, 'ntl-train-lora-qwen25-instyle/lora_out/job_0/COMPLETE.json': {'bytes': 101, 'sha256': '89d8f698ab817c01a8ef074edbc09b34ef67ec55a4eb27f0fbc8b791029dfec3'}, 'ntl-train-lora-qwen25-instyle/lora_out/job_1/COMPLETE.json': {'bytes': 101, 'sha256': '307a496a5b09e38e4b729b89f28ff9e432698b6c21a97d22283a49cf23e482e5'}, 'ntl-train-lora-qwen25-instyle/lora_out/job_2/COMPLETE.json': {'bytes': 101, 'sha256': 'bba61fc10fa0883bdce1bc40f82d19ce065f81522c0fbe0ec8bb9a85984ca35f'}, 'ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json': {'bytes': 88, 'sha256': 'ed24f3ec9e73878b99ee97730591ad17ebdff69e4f59e3f3dee90ea29d8ded8d'}, 'ntl-train-lora-qwen25-instyle/lora_out/job_final/adapter/adapter_config.json': {'bytes': 1284, 'sha256': '01f4767094380c58a8092eae287325c7b97f47905132076776d60afbe8e6af8e'}, 'ntl-train-lora-qwen25-instyle/lora_out/job_final/adapter/adapter_model.safetensors': {'bytes': 115378992, 'sha256': '509626893eb960bf6ce0fc9075c7a3c196ffa4994aba46dcf44508b8ab8c646b'}, 'ntl-train-lora-qwen25-instyle/lora_out/job_final/norms.safetensors': {'bytes': 230616, 'sha256': '29b8845f58771d6a0756cf66e3b303110b53e7e8f5939013706abf93e2db50ec'}, 'ntl-train-lora-qwen25-instyle/lora_out/job_final/tokenizer/tokenizer.json': {'bytes': 11421990, 'sha256': '0bf86e7bd1b719430e67169321519a1ba441a691eb763fd7abb0dc305eb41120'}, 'ntl-train-lora-qwen25-instyle/lora_out/job_final/tokenizer/tokenizer_config.json': {'bytes': 696, 'sha256': '5f007d04324afa93876e2a484190e868136c69dfe0e1a29a1063eae583df9d32'}, 'ntl-two-stage-lora/lora14b_v1/adapter/adapter_config.json': {'bytes': 1394, 'sha256': '28574e93c9fe2fda535f59f5b4723185ad418d39d59a48650c0cd48b7404fce5'}, 'ntl-two-stage-lora/lora14b_v1/adapter/adapter_model.safetensors': {'bytes': 173456688, 'sha256': '79952085c422373803acc7339544e6775c1d55eaca5ccd82492edd90a975eec1'}, 'ntl-two-stage-lora/lora14b_v1/heads.safetensors': {'bytes': 6227664, 'sha256': '43edc4e808b9d0114fbdbfe2fdaae307f0f99e07896b1d45e8548aeec310d089'}, 'ntl-two-stage-lora/lora14b_v1/meta.json': {'bytes': 931, 'sha256': 'ef274e94503adb1c5a4910dc3002a4595d63c525dac11d37a973ec2343de0c5e'}, 'ntl-two-stage-lora/lora8b_v1/adapter/adapter_config.json': {'bytes': 1393, 'sha256': 'cd1f40e92124ea643f934c4c3431385b11f37681f5de30a437b8022124c23043'}, 'ntl-two-stage-lora/lora8b_v1/adapter/adapter_model.safetensors': {'bytes': 116434760, 'sha256': '25d56499c7888dd1e274a92f35efd6acadf06349c928944c8cbee6eb56aabc4d'}, 'ntl-two-stage-lora/lora8b_v1/heads.safetensors': {'bytes': 4982480, 'sha256': 'aedeb224fe743f2a9f1e2f3344208d27e3fdc9f99df5a084f3280baee383a8b4'}, 'ntl-two-stage-lora/lora8b_v1/meta.json': {'bytes': 958, 'sha256': 'd84edb29b2c1c3b555a366dcb43df5661faed3a5b12abaf0f18e96cdf8aa5cda'}}
_copy_verified_inputs = 0
for _copy_relative, _copy_expected in _COPY_EXPECTED_MIRROR.items():
    _copy_suffix = '/ntl-public-heads/' + _copy_relative
    _copy_matches = [record for path, record in _ntl_inputs.items() if path.endswith(_copy_suffix)]
    if len(_copy_matches) != 1:
        raise FileNotFoundError(f'Expected exactly one mirrored model input: {_copy_relative}')
    if _copy_matches[0] != _copy_expected:
        raise ValueError(f'Mirror differs from the scored source fingerprint: {_copy_relative}')
    _copy_verified_inputs += 1
print(f'Working-copy input verification passed: {_copy_verified_inputs} model fingerprints match.', flush=True)


In [ ]:
from pathlib import Path
_PREPARED_FILES = {'hybrid_math.py': '"""Probability alignment, conservative blending and bounded prompt grouping."""\nimport numpy as np\n\n\ndef validate_probabilities(probability, rows, classes):\n    p = np.asarray(probability, dtype=np.float64)\n    if (p.shape != (rows, classes) or not np.isfinite(p).all() or (p < 0).any()\n            or not np.allclose(p.sum(1), 1.0, atol=1e-5, rtol=0)):\n        raise ValueError(\'Invalid probability shape, values or normalization\')\n    return p\n\n\ndef softmax(values):\n    z = np.asarray(values, dtype=np.float64)\n    z = np.exp(z - z.max(1, keepdims=True))\n    return z / z.sum(1, keepdims=True)\n\n\ndef align_probabilities(probability, actual_labels, expected_labels):\n    if len(set(actual_labels)) != len(actual_labels) or set(actual_labels) != set(expected_labels):\n        raise ValueError(\'Checkpoint label set differs from the competition\')\n    p = validate_probabilities(probability, len(probability), len(actual_labels))\n    return p[:, [actual_labels.index(label) for label in expected_labels]]\n\n\ndef mix_candidates(baseline, candidates, weights):\n    base = validate_probabilities(baseline, len(baseline), baseline.shape[1])\n    if set(candidates) - set(weights) or any(w < 0 or not np.isfinite(w) for w in weights.values()):\n        raise ValueError(\'Unknown candidate or invalid blend weight\')\n    if sum(weights.values()) > 1:\n        raise ValueError(\'Candidate weights exceed the probability mass\')\n    used = {name: weights[name] for name in candidates}\n    answer = (1.0 - sum(used.values())) * base\n    for name, weight in used.items():\n        answer += weight * validate_probabilities(candidates[name], len(base), base.shape[1])\n    return validate_probabilities(answer, len(base), base.shape[1]), {\'baseline\':1-sum(used.values()), **used}\n\n\ndef sinkhorn(probability, iterations=100):\n    p = np.clip(np.asarray(probability, dtype=np.float64), 1e-12, 1.0)\n    p /= p.sum(1, keepdims=True)\n    if len(p) < 50:\n        return p\n    for _ in range(iterations):\n        p *= (len(p) / p.shape[1]) / p.sum(0, keepdims=True)\n        p /= p.sum(1, keepdims=True)\n    return validate_probabilities(p, len(p), p.shape[1])\n\n\ndef content_groups(texts, threshold=0.85):\n    from scipy.cluster.hierarchy import linkage, fcluster\n    from scipy.spatial.distance import squareform\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    if len(texts) < 2:\n        return np.arange(len(texts), dtype=np.int64)\n    try:\n        x = TfidfVectorizer(stop_words=\'english\', sublinear_tf=True).fit_transform(texts)\n    except ValueError:\n        return np.arange(len(texts), dtype=np.int64)\n    distance = np.clip(1-(x @ x.T).toarray(), 0, 1)\n    np.fill_diagonal(distance, 0)\n    return fcluster(linkage(squareform(distance, checks=False), method=\'average\'),\n                    threshold, criterion=\'distance\').astype(np.int64)\n\n\ndef checkpoint_groups_eligible(groups):\n    _, inverse, counts = np.unique(groups, return_inverse=True, return_counts=True)\n    return (counts[inverse] >= 2) & (counts[inverse] <= 18)\n\n\ndef selective_atlas_rows(probability, margin=0.3):\n    p = validate_probabilities(probability, len(probability), probability.shape[1])\n    if len(p) < 50:\n        return np.arange(len(p))\n    top = np.sort(p,axis=1)[:,-2:]\n    return np.flatnonzero(top[:,1]-top[:,0] < margin)\n\n\ndef dense_group_hungarian(probability,groups):\n    """Dhruv-style hard assignment only in detected groups of 12 through 18 rows."""\n    from scipy.optimize import linear_sum_assignment\n    p=validate_probabilities(probability,len(probability),probability.shape[1])\n    groups=np.asarray(groups)\n    if groups.shape!=(len(p),):raise ValueError(\'Prompt group row count differs\')\n    predictions=p.argmax(1).copy()\n    decisions=[]\n    for group in np.unique(groups):\n        rows=np.flatnonzero(groups==group)\n        if not 12<=len(rows)<=min(18,p.shape[1]):continue\n        r,c=linear_sum_assignment(-np.log(np.clip(p[rows],1e-12,1)))\n        changed=int((predictions[rows[r]]!=c).sum())\n        predictions[rows[r]]=c\n        decisions.append({\'group\':int(group),\'size\':len(rows),\'changed_rows\':changed})\n    return predictions,{\'method\':\'dhruv-style-dense-group-hungarian\',\'minimum_group_size\':12,\n                        \'maximum_group_size\':18,\'changed_rows\':int((predictions!=p.argmax(1)).sum()),\n                        \'groups\':decisions}\n\n\ndef merge_pools(frames, train, labels):\n    import pandas as pd\n    import re\n    norm = lambda value: re.sub(r\'\\s+\', \' \', str(value).strip())\n    data = pd.concat([frame[[\'text\',\'model\']] for frame in frames],ignore_index=True)\n    data = data.dropna(subset=[\'text\',\'model\'])\n    data = data[data.model.isin(labels)].copy()\n    data[\'text\'] = data.text.map(norm)\n    data = data[data.text.ne(\'\')]\n    conflict = data.groupby(\'text\').model.nunique()\n    conflicts = set(conflict[conflict>1].index)\n    data = data[~data.text.isin(conflicts)].drop_duplicates(subset=[\'text\'])\n    data = data[~data.text.isin(set(train.text.map(norm)))].reset_index(drop=True)\n    if data.empty or set(data.model) != set(labels):\n        raise ValueError(\'External pool is empty or lacks competition classes\')\n    return data, {\'rows\':len(data), \'conflicting_texts_removed\':len(conflicts), \'classes\':len(labels)}\n', 'candidate_runtime.py': '"""Run one independent pretrained adapter or the Vishak-style SVC in a child process."""\nimport argparse\nimport glob\nimport gc\nimport json\nfrom pathlib import Path\nimport time\nimport numpy as np\nimport pandas as pd\nfrom hybrid_math import softmax, validate_probabilities, merge_pools, checkpoint_groups_eligible\n\nLABELS = sorted([\'gpt-5.5\',\'gpt-5.6-luna\',\'gpt-5.6-sol\',\'gpt-5.6-terra\',\'gpt-6-astra\',\n    \'gemini-3.1-pro-preview\',\'gemini-3.6-flash\',\'gemini-3.7-flash\',\'gemini-3.8-flash\',\n    \'claude-fable-5\',\'claude-opus-4-6\',\'claude-opus-4-8\',\'claude-opus-5\',\'claude-sonnet-5\',\n    \'qwen3.8-max\',\'glm-5.3\',\'glm-5.3-flash\',\'kimi-k3\'])\nLAYERS = [2,4,6,8,12,16,20,24]\nSPECS = {\n    \'ts_q3b14\':dict(model_type=\'qwen3\',hidden_size=5120,num_hidden_layers=40,keep=27,bs=8,gauss=0.20),\n    \'ts_q3b8\':dict(model_type=\'qwen3\',hidden_size=4096,num_hidden_layers=36,keep=24,bs=16,gauss=0.15),\n    \'ts_llama8\':dict(model_type=\'llama\',hidden_size=4096,num_hidden_layers=32,keep=24,bs=16,gauss=0.0),\n    \'ts_q25_7b\':dict(model_type=\'qwen2\',hidden_size=3584,num_hidden_layers=28,keep=24,bs=16,gauss=0.0),\n}\n\n\ndef find_base(root,spec):\n    from checkpoint_resolver import _ntl_has_model_weights\n    found=[]\n    for cfg in sorted({Path(p).resolve() for p in glob.glob(str(Path(root)/\'**/config.json\'),recursive=True)}):\n        try: data=json.loads(cfg.read_text())\n        except (OSError,ValueError): continue\n        if all(data.get(key)==spec[key] for key in [\'model_type\',\'hidden_size\',\'num_hidden_layers\']):\n            if _ntl_has_model_weights(str(cfg.parent)):found.append(cfg.parent)\n    if not found:raise FileNotFoundError(f\'No weighted base checkpoint for {spec}\')\n    return str(found[0])\n\n\ndef svc_predict(train,test,root):\n    import re\n    from scipy.sparse import hstack\n    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS,TfidfVectorizer\n    from sklearn.svm import LinearSVC\n    paths=[]\n    for name,dataset,owner in [(\'pool_v1.parquet\',\'ntl-two-stage-lora\',\'nishantdahal\'),\n                                (\'nlm_pool_v2.parquet\',\'nlm-matched-clean\',\'dhruvtibarewal\')]:\n        matches=sorted({Path(p).resolve() for p in glob.glob(str(Path(root)/\'**\'/name),recursive=True)})\n        preferred=[p for p in matches if p.parent.name==dataset and p.parent.parent.name==owner]\n        if len(preferred)==1:matches=preferred\n        if not matches:raise FileNotFoundError(f\'No accessible {name}; root directories: {[p.name for p in Path(root).iterdir()]}\')\n        if len(matches)>1:\n            import hashlib\n            hashes=[]\n            for path in matches:\n                with path.open(\'rb\') as stream:hashes.append(hashlib.file_digest(stream,\'sha256\').hexdigest())\n            if len(set(hashes))>1:raise ValueError(f\'Ambiguous, different pools for {name}: {matches}\')\n            matches=matches[:1]\n        paths.append(matches[0])\n    pool,report=merge_pools([pd.read_parquet(path) for path in paths],train,LABELS)\n    # Reproduction of the public Vishak SVC recipe, with our accessible merged pool.\n    shape=lambda text:re.sub(r\'[^\\W\\d_]+\',lambda m:\'X\' if m[0].isupper() else \'x\',text)\n    function=lambda text:\' \'.join(word.lower() if word.lower() in ENGLISH_STOP_WORDS or not word.isalnum() else \'C\'\n                                  for word in re.findall(r\'\\w+|[^\\w\\s]\',text))\n    views=[(str,\'char\',(1,5),400000,1.0),(str,\'word\',(1,3),300000,0.7),\n           (shape,\'char\',(2,6),80000,0.5),(function,\'word\',(1,4),100000,0.6)]\n    texts=pool.text.tolist()+train.text.tolist()\n    a,b=[],[]\n    for transform,analyzer,ngrams,maximum,weight in views:\n        options=dict(analyzer=analyzer,ngram_range=ngrams,max_features=maximum,\n                     lowercase=False,sublinear_tf=True,min_df=3,dtype=np.float32)\n        if analyzer==\'word\':options[\'token_pattern\']=r\'(?u)\\b\\w+\\b|[^\\w\\s]\'\n        vectorizer=TfidfVectorizer(**options)\n        combined=vectorizer.fit_transform([transform(t) for t in texts+test.text.tolist()])\n        a.append(combined[:len(texts)]*weight)\n        b.append(combined[len(texts):]*weight)\n        print(\'SVC view\',analyzer,ngrams,\'ready\',flush=True)\n        del combined\n    mapping={label:i for i,label in enumerate(LABELS)}\n    y=np.r_[pool.model.map(mapping).values,train.model.map(mapping).values]\n    sample_weight=np.r_[np.full(len(pool),0.5),np.ones(len(train))]\n    totals=np.bincount(y,weights=sample_weight,minlength=len(LABELS))\n    sample_weight*=totals.mean()/totals[y]\n    classifier=LinearSVC(C=0.5,max_iter=3000,random_state=42)\n    scores=classifier.fit(hstack(a).tocsr(),y,sample_weight=sample_weight).decision_function(hstack(b).tocsr())\n    if classifier.classes_.tolist()!=list(range(len(LABELS))):raise ValueError(\'SVC class order differs\')\n    return softmax(scores),report\n\n\ndef adapter_predict(name,train,test,groups,root,skip_gaussian=False,checkpoint=None):\n    import torch\n    from safetensors.torch import load_file\n    from transformers import AutoConfig,AutoModel,AutoTokenizer\n    spec=SPECS[name]\n    bases=find_base(root,spec)\n    adapters=sorted({Path(p).resolve().parent for p in glob.glob(str(Path(root)/\'**\'/name/\'heads.pt\'),recursive=True)\n                     if Path(p).parent.parent.name==\'nlm-l24-adapters\'})\n    if len(adapters)!=1:raise FileNotFoundError(f\'Expected one pinned Dhruv adapter {name}: {adapters}\')\n    adapter=adapters[0]\n    import hashlib\n    expected = {\'ts_q3b8\': \'fb580805fd4bf4e6cfed9bcbb690dea45d342f02571366c86694cb2ffe769b2b\', \'ts_q3b14\': \'36ac37a42fd8421188a66df27445dddb9bdd0e89afd6e66c2bcf04ab42ea8019\', \'ts_llama8\': \'616d6fe1bca2d1eba8548bb25d494362e24c15f1c1faa459bc62a531f8fe6fab\', \'ts_q25_7b\': \'57f6372d3a50e660279db93193f84db1cdfb3ea9a677d7e76a66682fdb5cb261\'}\n    with (adapter/\'heads.pt\').open(\'rb\') as stream:\n        if hashlib.file_digest(stream,\'sha256\').hexdigest()!=expected[name]:\n            raise ValueError(\'Classifier head differs from our checked pinned asset\')\n    if name==\'ts_llama8\':\n        manifest_path=Path(bases)/\'hf-model-manifest.json\'\n        if not manifest_path.is_file():raise FileNotFoundError(\'Verified HF base manifest is required for Llama\')\n        manifest=json.loads(manifest_path.read_text())\n        if (manifest[\'repo\']!=\'unsloth/Meta-Llama-3.1-8B\' or\n                manifest[\'revision\']!=\'e9a141a2091ea561b96483212645a2a05e6f99fc\' or\n                not manifest[\'all_weight_shards_verified_against_hf\'] or manifest[\'weight_shards\']!=4):\n            raise ValueError(\'Llama base provenance differs from the checked adapter base\')\n        for file,record in manifest[\'files\'].items():\n            path=Path(bases)/file\n            if not path.is_file() or path.stat().st_size!=record[\'bytes\']:\n                raise ValueError(\'Llama base asset missing or truncated: \'+file)\n    head=torch.load(adapter/\'heads.pt\',map_location=\'cpu\',weights_only=True)\n    cfg=AutoConfig.from_pretrained(bases)\n    cfg.num_hidden_layers,cfg.use_cache=spec[\'keep\'],False\n    if getattr(cfg,\'layer_types\',None) is not None:cfg.layer_types=cfg.layer_types[:spec[\'keep\']]\n    tokenizer=AutoTokenizer.from_pretrained(bases)\n    tokenizer.padding_side=\'right\'\n    if tokenizer.pad_token is None:tokenizer.pad_token=tokenizer.eos_token\n    memory={i:\'13GiB\' for i in range(torch.cuda.device_count())}\n    model,loading_info=AutoModel.from_pretrained(bases,config=cfg,dtype=torch.float16,\n        device_map=\'auto\',max_memory=memory,output_loading_info=True)\n    if loading_info.get(\'missing_keys\') or loading_info.get(\'mismatched_keys\') or loading_info.get(\'error_msgs\'):\n        raise RuntimeError(\'Base checkpoint has missing or mismatched weights; refusing random initialization\')\n    model.eval()\n    modules=dict(model.named_modules())\n    config=json.loads((adapter/\'adapter_config.json\').read_text())\n    state=load_file(str(adapter/\'adapter_model.safetensors\'))\n    merged=0\n    with torch.no_grad():\n        for key,a in state.items():\n            if \'.lora_A.\' not in key:continue\n            module_name=key.replace(\'base_model.model.\',\'\',1).split(\'.lora_A.\')[0]\n            if module_name not in modules and \'model.\'+module_name in modules:module_name=\'model.\'+module_name\n            if module_name not in modules:raise KeyError(f\'Adapter module absent: {module_name}\')\n            w=modules[module_name].weight\n            b=state[key.replace(\'.lora_A.\',\'.lora_B.\')]\n            w.add_(((b.to(w.device,torch.float32)@a.to(w.device,torch.float32))*config[\'lora_alpha\']/config[\'r\']).to(w.dtype))\n            merged+=1\n    if merged!=spec[\'keep\']*7:raise ValueError(f\'Unexpected merged module count: {merged}\')\n    del state,modules\n    hidden=spec[\'hidden_size\']\n    norms=[torch.nn.LayerNorm(hidden) for _ in LAYERS]\n    for i,norm in enumerate(norms):\n        norm.load_state_dict({\'weight\':head[f\'norms.{i}.weight\'],\'bias\':head[f\'norms.{i}.bias\']})\n    expected=(len(LABELS),hidden*len(LAYERS))\n    if tuple(head[\'plain.weight\'].shape)!=expected or tuple(head[\'centered.weight\'].shape)!=expected:\n        raise ValueError(\'Classifier head dimension differs from the adapter\')\n    device=model.get_input_embeddings().weight.device\n    def features(texts,active_model=model):\n        # Match Dhruv\'s recipe: tokenizer defaults, no added EOS, max_len 384, mask first token.\n        token_ids=tokenizer(list(texts),truncation=True,max_length=384)[\'input_ids\']\n        order=np.argsort([len(item) for item in token_ids],kind=\'stable\')\n        result=np.zeros((len(texts),hidden*len(LAYERS)),np.float32)\n        batch_start=0\n        batch_size=spec[\'bs\']\n        while batch_start<len(order):\n            indexes=order[batch_start:batch_start+batch_size]\n            pack=tokenizer.pad({\'input_ids\':[token_ids[i] for i in indexes]},return_tensors=\'pt\').to(device)\n            try:\n                with torch.inference_mode():\n                    states=active_model(**pack,output_hidden_states=True,use_cache=False).hidden_states\n                    mask=pack[\'attention_mask\'].cpu().unsqueeze(-1).float()\n                    mask[:,0]=0\n                    den=mask.sum(1).clamp(min=1)\n                    if not all(torch.isfinite(states[layer]).all() for layer in LAYERS):\n                        raise FloatingPointError(\'Nonfinite pretrained adapter activations\')\n                    result[indexes]=torch.cat([norm((states[layer].float().cpu()*mask).sum(1)/den)\n                                                for layer,norm in zip(LAYERS,norms)],-1).numpy()\n                del states,pack\n            except torch.cuda.OutOfMemoryError:\n                del pack\n                torch.cuda.empty_cache()\n                if batch_size==1:raise\n                batch_size=max(1,batch_size//2)\n                print(\'Reduced batch size to\',batch_size,flush=True)\n                continue\n            batch_start+=len(indexes)\n            if batch_start%256<batch_size:print(name,\'features\',batch_start,\'/\',len(order),flush=True)\n        return result\n    z=features(test.text.tolist())\n    logits=z@head[\'plain.weight\'].float().numpy().T+head[\'plain.bias\'].float().numpy()\n    lp=np.log(np.clip(softmax(logits),1e-12,1))\n    eligible=checkpoint_groups_eligible(groups)\n    if eligible.any():\n        centred=z[eligible].copy()\n        local=groups[eligible]\n        for group in np.unique(local):centred[local==group]-=centred[local==group].mean(0)\n        twin=centred@head[\'centered.weight\'].float().numpy().T+head[\'centered.bias\'].float().numpy()\n        lp[eligible]+=0.35*np.log(np.clip(softmax(twin),1e-12,1))\n    gaussian_applied=False\n    if checkpoint is not None:\n        checkpoint(softmax(lp),{\'adapter\':name,\'base\':bases,\'merged_modules\':merged,\n            \'full_data_adapter\':True,\'qwen_lora_training\':False,\'gaussian_applied\':False,\n            \'grouped_rows\':int(eligible.sum()),\'pre_gaussian_checkpoint\':True})\n    if spec[\'gauss\']>0 and eligible.any() and not skip_gaussian:\n        from gauss import fit as original_fit,scores\n        from cache_runtime import get_store\n        fit=lambda X,y,prompts,cfg=None:get_store().fit(\'dhruv\',original_fit,X,y,prompts,cfg)\n        from cache_runtime import dhruv_train_features\n        ztrain=dhruv_train_features(name,spec,bases,adapter)\n        y=train.model.map({label:i for i,label in enumerate(LABELS)}).to_numpy(dtype=int)\n        params=fit(ztrain,y,train.prompt_id.values,dict(red=\'wpca\',d=128,se_shrink=\'lw\',sp_shrink=\'lw\',\n                   lam_floor=1e-4,n_exact=3,inj=True,iters=5,repel=True,sp_scale=1.0))\n        params[\'T1\']=params[\'Tg\']=3.0\n        local=np.unique(groups[eligible],return_inverse=True)[1]\n        gaussian=scores(z[eligible],local,params)\n        if not np.isfinite(gaussian).all():raise ValueError(\'Nonfinite Gaussian scores\')\n        lp[eligible]+=spec[\'gauss\']*gaussian\n        gaussian_applied=True\n    report={\'adapter\':name,\'base\':bases,\'merged_modules\':merged,\'full_data_adapter\':True,\n            \'qwen_lora_training\':False,\'gaussian_applied\':gaussian_applied,\'grouped_rows\':int(eligible.sum()),\n            \'gaussian_skipped_for_budget\':bool(skip_gaussian and spec[\'gauss\']>0 and eligible.any())}\n    probability=softmax(lp)\n    del features,model\n    gc.collect();torch.cuda.empty_cache()\n    return probability,report\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'member\',choices=[\'svc\']+list(SPECS))\n    parser.add_argument(\'--request\',required=True)\n    parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--root\',default=\'/kaggle/input\')\n    parser.add_argument(\'--skip-gaussian\',action=\'store_true\')\n    args=parser.parse_args()\n    start=time.time()\n    request=json.loads(Path(args.request).read_text())\n    train=pd.read_csv(request[\'train_path\'])\n    test=pd.read_csv(request[\'test_path\'])\n    for frame in [train,test]:frame[\'text\']=frame.text.fillna(\'\').astype(str)\n    with np.load(request[\'group_path\'],allow_pickle=False) as values:\n        if values[\'ids\'].tolist()!=test.id.astype(str).tolist():raise ValueError(\'Group row order differs\')\n        groups=values[\'groups\']\n    def checkpoint(p,report):\n        path=args.output+\'.plain.npz\'\n        np.savez_compressed(path,probabilities=p,labels=np.asarray(LABELS),ids=test.id.astype(str).to_numpy(dtype=str))\n        Path(path+\'.json\').write_text(json.dumps(report,indent=2))\n    if args.member==\'svc\':p,report=svc_predict(train,test,args.root)\n    else:p,report=adapter_predict(args.member,train,test,groups,args.root,args.skip_gaussian,checkpoint)\n    validate_probabilities(p,len(test),len(LABELS))\n    np.savez_compressed(args.output,probabilities=p,labels=np.asarray(LABELS),ids=test.id.astype(str).to_numpy(dtype=str))\n    report.update({\'seconds\':time.time()-start,\'rows\':len(test),\'member\':args.member})\n    Path(args.output+\'.json\').write_text(json.dumps(report,indent=2))\n    print(json.dumps(report),flush=True)\n', 'atlas_runtime.py': '"""Budgeted ATLAS inference with validated partial checkpoints for completed rows."""\nimport argparse,gc,json,sys,time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom safetensors.torch import load_file\nfrom candidate_runtime import find_base\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--request\',required=True)\n    parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--seconds\',type=float,required=True)\n    args=parser.parse_args()\n    deadline=time.time()+args.seconds\n    req=json.loads(Path(args.request).read_text())\n    frame=pd.read_csv(req[\'test_path\'])\n    frame.text=frame.text.fillna(\'\').astype(str)\n    wanted=np.load(req[\'atlas_rows_path\'],allow_pickle=False)\n    roots=list(Path(\'/kaggle/input\').rglob(\'atlas_bundle.json\'))\n    if len(roots)!=1:raise ValueError(\'Exactly one ATLAS bundle must be attached\')\n    root=roots[0].parent\n    meta=json.loads(roots[0].read_text())\n    import hashlib\n    for name,digest in meta[\'files\'].items():\n        with (root/name).open(\'rb\') as handle:\n            if hashlib.file_digest(handle,\'sha256\').hexdigest()!=digest:raise ValueError(\'Changed ATLAS asset\')\n    sys.path.insert(0,str(root))\n    from ntl_atlas.model import load_model\n    from ntl_rlcd.data import LABELS\n    base=find_base(\'/kaggle/input\',dict(model_type=\'qwen3\',hidden_size=4096,num_hidden_layers=36))\n    model,tokenizer=load_model(base,LABELS,meta[\'training_config\'],device=\'auto\')\n    model.load_trainable(load_file(str(root/\'adapters.safetensors\')))\n    model.eval()\n    ids=tokenizer(frame.text.iloc[wanted].tolist(),truncation=True,max_length=meta[\'training_config\'][\'max_length\'])[\'input_ids\']\n    order=np.argsort([len(item) for item in ids],kind=\'stable\')\n    complete,values=[],[]\n    batch_size=2\n    pos=0\n    while pos<len(order) and time.time()<deadline-60:\n        chosen=order[pos:pos+batch_size]\n        pack=tokenizer.pad({\'input_ids\':[ids[i] for i in chosen]},return_tensors=\'pt\').to(model.device)\n        try:\n            with torch.inference_mode():p,_=model.score(pack[\'input_ids\'],pack[\'attention_mask\'],source_chunk=1)\n            p=p.float().cpu().numpy()\n            if not np.isfinite(p).all() or not np.allclose(p.sum(1),1,atol=1e-5):raise ValueError(\'Invalid ATLAS output\')\n            complete.extend(wanted[chosen].tolist());values.extend(p.tolist())\n            pos+=len(chosen)\n            del p,pack\n        except torch.cuda.OutOfMemoryError:\n            del pack\n            gc.collect();torch.cuda.empty_cache()\n            if batch_size==1:raise\n            batch_size=1\n            continue\n        # Atomic file replacement leaves a usable prefix if the child reaches its timeout.\n        temporary=args.output+\'.tmp.npz\'\n        np.savez_compressed(temporary,rows=np.asarray(complete,dtype=int),probabilities=np.asarray(values),\n                            labels=np.asarray(LABELS),ids=frame.id.astype(str).iloc[complete].to_numpy(dtype=str))\n        Path(temporary).replace(args.output)\n        if pos%20<batch_size:print(\'ATLAS completed\',pos,\'/\',len(order),flush=True)\n    Path(args.output+\'.json\').write_text(json.dumps({\'requested\':len(wanted),\'completed\':len(complete),\n        \'stopped_for_time_budget\':pos<len(order)},indent=2))\n', 'baseline_runtime.py': 'from cache_runtime import strict_pretrained_loading\nstrict_pretrained_loading()\n"""Reject config-only adapter folders when selecting a real base checkpoint."""\nimport json as _weights_json\nimport os as _weights_os\n\n\ndef _ntl_has_model_weights(directory):\n    for name in (\'model.safetensors\', \'pytorch_model.bin\'):\n        if _weights_os.path.isfile(_weights_os.path.join(directory, name)):\n            return True\n    for name in (\'model.safetensors.index.json\', \'pytorch_model.bin.index.json\'):\n        path = _weights_os.path.join(directory, name)\n        if not _weights_os.path.isfile(path):\n            continue\n        try:\n            with open(path, encoding=\'utf-8\') as stream:\n                mapping = _weights_json.load(stream).get(\'weight_map\', {})\n            shards = set(mapping.values())\n            if shards and all(isinstance(shard, str) and\n                              _weights_os.path.isfile(_weights_os.path.join(directory, shard))\n                              for shard in shards):\n                return True\n        except (OSError, ValueError, TypeError, AttributeError):\n            continue\n    return False\n\n\nimport gc, glob, json, os, re, time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\n\nT0 = time.time()\nINPUT_DIR = "/kaggle/input"\n\n# Blend weights, fit on out-of-fold predictions (log probabilities are multiplied by these and summed).\nW_SVC = 1.021\nLORAS = {  # checkpoint folder: blend weight, centred-twin weight, batch size, base model to look for\n    "lora8b_v1":  dict(w=0.226, twin_w=0.079, bs=16, base=dict(model_type="qwen3", hidden_size=4096, num_hidden_layers=36)),\n    "lora14b_v1": dict(w=0.429, twin_w=0.150, bs=8,  base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),\n}\nTWIN = dict(thresh=0.85, min_size=2, max_size=18)   # topic clusters that get the centred twin\n\n# Column order of the checkpoints\' heads (checked against each checkpoint\'s meta.json below).\nLABELS = [\'gpt-5.5\', \'gpt-5.6-luna\', \'gpt-5.6-sol\', \'gpt-5.6-terra\', \'gpt-6-astra\',\n          \'gemini-3.1-pro-preview\', \'gemini-3.6-flash\', \'gemini-3.7-flash\', \'gemini-3.8-flash\',\n          \'claude-opus-4-6\', \'claude-opus-4-8\', \'claude-opus-5\', \'claude-sonnet-5\', \'claude-fable-5\',\n          \'qwen3.8-max\', \'glm-5.3\', \'glm-5.3-flash\', \'kimi-k3\']\nK = len(LABELS)\n\n\ndef find(pattern):\n    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))\n    if not hits:\n        raise FileNotFoundError(pattern)\n    return hits[0]\n\n\ndef softmax(x):\n    e = np.exp(x - x.max(1, keepdims=True))\n    return e / e.sum(1, keepdims=True)\n\n\n_competition_dir = next(p for p in ["/kaggle/input/competitions/name-that-llm", "/kaggle/input/name-that-llm"] if os.path.exists(p + "/train.csv"))\ntrain = pd.read_csv(_competition_dir + "/train.csv")\ntest = pd.read_csv(_competition_dir + "/test.csv")\ntest["text"] = test["text"].fillna("").astype(str)\ny = train["model"].map({l: i for i, l in enumerate(LABELS)}).values\nLOGIT = np.zeros((len(test), K))   # every member adds w * log p here\nprint("train", train.shape, "test", test.shape, flush=True)\n\n\nfrom scipy.sparse import hstack\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.svm import LinearSVC\n\nFUNCTION_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())\nVIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("function", 1, 3, 50000, 0.5)]\n\n\ndef shape_text(t):\n    return re.sub(r"[^\\W\\d_]+", lambda m: "X" if m[0].isupper() else "x", t)\n\n\ndef function_text(t):\n    return " ".join(w.lower() if w.lower() in FUNCTION_WORDS or not w.isalnum() else "CONTENT"\n                    for w in re.findall(r"\\w+|[^\\w\\s]", t))\n\n\npool = pd.read_parquet(find("pool_v1.parquet"))\nfit_texts = train.text.astype(str).tolist() + pool.text.tolist()\nfit_y = np.r_[y, pool.model.map({l: i for i, l in enumerate(LABELS)}).values]\nA, B = [], []\nfor view, lo, hi, max_features, weight in VIEWS:\n    prep = shape_text if view == "shape" else function_text if view == "function" else (lambda s: s)\n    analyzer = "char" if view == "shape" else "word" if view == "function" else view\n    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,\n                max_features=max_features, dtype=np.float32)\n    if analyzer == "word":\n        opts["token_pattern"] = r"(?u)\\b\\w+\\b|[^\\w\\s]"\n    vec = TfidfVectorizer(**opts)\n    A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)\n    B.append(vec.transform([prep(t) for t in test.text]) * weight)\nsvc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(hstack(A, format="csr"), fit_y)\nP_svc = softmax(svc.decision_function(hstack(B, format="csr")))\nLOGIT += W_SVC * np.log(np.clip(P_svc, 1e-9, 1))\ndel A, B, svc, pool\ngc.collect()\nprint(f"n-gram model done ({time.time() - T0:.0f}s)", flush=True)\n\n\nfrom safetensors.torch import load_file\nfrom transformers import AutoConfig, AutoModel, AutoTokenizer\n\n\ndef find_base(spec):\n    for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):\n        try:\n            c = json.load(open(cfg))\n        except Exception:\n            continue\n        if all(c.get(k) == v for k, v in spec.items()) and _ntl_has_model_weights(os.path.dirname(cfg)):\n            return os.path.dirname(cfg)\n    raise FileNotFoundError(f"attach the base model {spec}")\n\n\ndef load_trunk(base_dir, keep, dtype):\n    """First `keep` decoder layers, split over both T4s when it does not fit on one."""\n    mc = AutoConfig.from_pretrained(base_dir)\n    mc.num_hidden_layers, mc.use_cache = keep, False\n    if getattr(mc, "layer_types", None) is not None:\n        mc.layer_types = mc.layer_types[:keep]\n    hd = getattr(mc, "head_dim", mc.hidden_size // mc.num_attention_heads)\n    gib = 2 * (mc.vocab_size * mc.hidden_size + keep * (2 * mc.hidden_size * mc.num_attention_heads * hd\n               + 2 * mc.hidden_size * mc.num_key_value_heads * hd + 3 * mc.hidden_size * mc.intermediate_size)) / 2 ** 30\n    if gib > 13 and torch.cuda.device_count() > 1:\n        mem = {i: "13GiB" for i in range(torch.cuda.device_count())}\n        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()\n    else:\n        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()\n    model.norm = nn.Identity()\n    return model\n\n\n@torch.no_grad()\ndef merge_lora(model, adapter_dir):\n    acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))\n    scale = acfg["lora_alpha"] / acfg["r"]\n    sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))\n    modules = dict(model.named_modules())\n    for k, A in sd.items():\n        if ".lora_A." in k:\n            name = k.split(".lora_A.")[0]\n            lin = modules[name[name.index("layers."):]]\n            Bm = sd[k.replace(".lora_A.", ".lora_B.")]\n            lin.weight.add_(((Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale).to(lin.weight.dtype))\n\n\n@torch.no_grad()\ndef lora_features(ckpt_dir, base_dir, texts, bs):\n    """Pooled features Z (n, 8 * hidden) and the checkpoint\'s heads."""\n    meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))\n    assert meta["labels"] == LABELS, "label order differs from the checkpoint\'s heads"\n    cfg = meta["cfg"]\n    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16\n    tok = AutoTokenizer.from_pretrained(base_dir)\n    tok.padding_side = "right"\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    model = load_trunk(base_dir, cfg["keep"], dtype)\n    merge_lora(model, os.path.join(ckpt_dir, "adapter"))\n    heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))\n    layers, H = cfg["layers"], model.config.hidden_size\n    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()\n    norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})\n    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]\n    ids = [x + [tok.eos_token_id] for x in ids]\n    order = np.argsort([len(x) for x in ids], kind="stable")   # similar lengths together = less padding\n    Z = np.zeros((len(ids), H * len(layers)), np.float32)\n    for s in range(0, len(order), bs):\n        idx = order[s:s + bs]\n        enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)\n        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\n        if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow: redo this batch in bf16\n            model.to(torch.bfloat16)\n            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\n            model.to(dtype)\n        m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")\n        m[:, 0] = 0   # the first token is an attention sink; leave it out of the mean\n        den = m.sum(1).clamp(min=1)\n        Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()\n    del model\n    torch.cuda.empty_cache()\n    return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}\n\n\nfrom scipy.cluster.hierarchy import fcluster, linkage\nfrom scipy.spatial.distance import squareform\n\nXt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test.text.tolist())\nD = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)\nnp.fill_diagonal(D, 0)\ngroups = fcluster(linkage(squareform(D, checks=False), method="average"), TWIN["thresh"], criterion="distance") \\\n    if len(test) >= 2 else np.ones(len(test), dtype=int)\nsize = np.bincount(groups)[groups]\ntwin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])\ndel D\nprint(f"topic clusters: {int((size == 1).sum())} singletons, {int(twin_rows.sum())} passages in clusters of 2-18", flush=True)\n\n# Gaussian shared-prompt core, credited to prvsiyan.\nimport itertools\nK = 18\nDEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,\n           "iters": 5, "repel": True, "sp_scale": 1.0}\n\n# =================================================================== primitives ====\ndef _lse(A, axis=None, keepdims=False):\n    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)\n    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m\n    if keepdims: return out\n    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())\n\ndef log_softmax(L): return L - _lse(L, axis=1, keepdims=True)\n\ndef _lw_shrinkage(X):\n    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""\n    from sklearn.covariance import ledoit_wolf_shrinkage\n    return float(ledoit_wolf_shrinkage(X, assume_centered=True))\n\ndef _shrunk(S, a):\n    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)\n\ndef _two_way(Z, y, pinv, G, iters=4):\n    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""\n    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)\n    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))\n    for _ in range(iters):\n        R = Z - P[pinv]\n        for c in cls: mu[c] = R[y == c].mean(0)\n        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]\n    return mu, P, cnt_p\n\ndef fit(X, y, prompts, cfg=None):\n    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.\n    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller\'s cross-fit)."""\n    c = dict(DEFAULT); c.update(cfg or {})\n    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape\n    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)\n    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx\n    if c["red"] == "none":\n        V = np.eye(D)\n    else:\n        if c["red"] == "wpca":\n            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]\n        elif c["red"] == "pca": R = Xs\n        else: raise ValueError(c["red"])\n        d = int(min(c["d"], N - G - 1, D))\n        # economy SVD through the (N x N) Gram matrix (N << D for the real views)\n        if N < D:\n            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]\n            V = (R.T @ U) / np.sqrt(w)\n        else:\n            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T\n    Z = Xs @ V; d = Z.shape[1]\n    mu, P, ng = _two_way(Z, y, pinv, G)\n    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)\n    Se_raw = E.T @ E / dof\n    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)\n    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw\n    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])\n    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]\n    # joint diagonalisation: A Se A\' = I, A Sp A\' = diag(lam)\n    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)\n    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li\n    W = V @ A.T\n    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,\n            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}\n\ndef transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]\n\n_MASK = {}\ndef _inj_mask(n, C):\n    if (n, C) not in _MASK:\n        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)\n        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]\n        _MASK[(n, C)] = bad\n    return _MASK[(n, C)]\n\ndef singleton_ll(U, m, lam):\n    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""\n    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)\n\ndef joint_terms(U, m, lam):\n    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""\n    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]\n    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)\n    return a, B\n\ndef joint_table(a, B):\n    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""\n    n, C = a.shape; Q = np.zeros([C] * n)\n    for i in range(n):\n        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)\n    for i, j in itertools.combinations(range(n), 2):\n        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)\n    return Q\n\ndef exact_marginals(Q, T=1.0, inj=True):\n    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""\n    n = Q.ndim; C = Q.shape[0]; Qt = Q / T\n    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)\n    lz = _lse(Qt); out = np.empty((n, C))\n    for i in range(n):\n        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz\n    return out\n\ndef em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):\n    """cavity EM for one group (any n >= 2): returns (n, C) log q."""\n    n = len(U); v = lam / (1 + (n - 1) * lam)\n    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))\n    q = np.exp(log_softmax(L))\n    for _ in range(iters):\n        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)\n        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T\n        if repel:   # mean-field at-most-once: discourage classes the other members already hold\n            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)\n        q = np.exp(log_softmax(L))\n    return log_softmax(L)\n\nLAM_DIFF = 1e6\ndef lam_g(prm, cfg=None):\n    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg \'diff\'): then the joint likelihood\n    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs\n    centred prototypes) and the prompt term cancels exactly."""\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})\n    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]\n\ndef group_logpost(U, prm, T1=None, Tg=None, cfg=None):\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)\n    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg\n    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)\n    if n <= c["n_exact"]:\n        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])\n    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])\n\ndef scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):\n    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""\n    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))\n    sizes = np.bincount(lab); s1 = sizes[lab] == 1\n    T1 = prm["T1"] if T1 is None else T1\n    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)\n    for g in np.where(sizes >= 2)[0]:\n        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)\n    return out\n\n\nfrom cache_runtime import install_main as _cache_install_main\n_cache_install_main(globals())\n\nfor ckpt_name, mc in LORAS.items():\n    ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))\n    if ckpt_name == \'lora8b_v1\':\n        # One 8B trunk load for official-train and test features; classifier checkpoints remain unchanged.\n        _all_texts = test.text.tolist() + train.text.astype(str).tolist()\n        _all_Z, heads = lora_features(ckpt, find_base(mc[\'base\']), _all_texts, mc[\'bs\'])\n        Z, _train_Z = _all_Z[:len(test)], _all_Z[len(test):]\n        _prm = fit(_train_Z, y, train[\'prompt_id\'].values,\n                   dict(red=\'wpca\', d=128, se_shrink=\'lw\', sp_shrink=\'lw\', lam_floor=1e-4,\n                        n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0))\n        _prm[\'T1\'] = _prm[\'Tg\'] = 3.0\n        if twin_rows.any():\n            _lp = scores(Z[twin_rows], np.unique(groups[twin_rows], return_inverse=True)[1], _prm)\n            assert np.isfinite(_lp).all()\n            LOGIT[twin_rows] += 0.10 * _lp\n            print(\'NEW_LORA_GAUSSIAN rows\', int(twin_rows.sum()), \'d128 T3 weight.10\', flush=True)\n        else:\n            print(\'NEW_LORA_GAUSSIAN fitted; no eligible public-test groups\', flush=True)\n        # Exercise exact pair inference even on the three-row example, without touching its predictions.\n        _pid = train[\'prompt_id\'].values\n        _ii = np.flatnonzero(_pid == _pid[0])[:2]\n        assert len(_ii) == 2\n        _qa = scores(_train_Z[_ii], np.zeros(2, dtype=int), _prm)\n        assert _qa.shape == (2, K) and np.isfinite(_qa).all() and np.allclose(np.exp(_qa).sum(1), 1)\n        print(\'NEW_LORA_GAUSSIAN exact-pair probability check PASS\', flush=True)\n        del _all_Z, _train_Z, _prm, _qa, _all_texts\n    else:\n        Z, heads = lora_features(ckpt, find_base(mc[\'base\']), test.text.tolist(), mc[\'bs\'])\n    LOGIT += mc["w"] * np.log(np.clip(softmax(Z @ heads["plain.weight"].T + heads["plain.bias"]), 1e-9, 1))\n    if twin_rows.any():\n        Zc = Z.copy()\n        for g in np.unique(groups[twin_rows]):\n            Zc[groups == g] -= Zc[groups == g].mean(0)\n        centred = Zc[twin_rows] @ heads["centered.weight"].T + heads["centered.bias"]\n        LOGIT[twin_rows] += mc["twin_w"] * np.log(np.clip(softmax(centred), 1e-9, 1))\n    del Z\n    gc.collect()\n    print(f"{ckpt_name} done ({time.time() - T0:.0f}s)", flush=True)\n\n\n_GAUSSIAN_SOURCE = \'"""Reject config-only adapter folders when selecting a real base checkpoint."""\\nimport json as _weights_json\\nimport os as _weights_os\\n\\n\\ndef _ntl_has_model_weights(directory):\\n    for name in (\\\'model.safetensors\\\', \\\'pytorch_model.bin\\\'):\\n        if _weights_os.path.isfile(_weights_os.path.join(directory, name)):\\n            return True\\n    for name in (\\\'model.safetensors.index.json\\\', \\\'pytorch_model.bin.index.json\\\'):\\n        path = _weights_os.path.join(directory, name)\\n        if not _weights_os.path.isfile(path):\\n            continue\\n        try:\\n            with open(path, encoding=\\\'utf-8\\\') as stream:\\n                mapping = _weights_json.load(stream).get(\\\'weight_map\\\', {})\\n            shards = set(mapping.values())\\n            if shards and all(isinstance(shard, str) and\\n                              _weights_os.path.isfile(_weights_os.path.join(directory, shard))\\n                              for shard in shards):\\n                return True\\n        except (OSError, ValueError, TypeError, AttributeError):\\n            continue\\n    return False\\n\\n\\n# ============================================================ 0. configuration\\nimport gc, glob, json, os, pathlib, re, time\\nimport numpy as np, pandas as pd\\ntry:\\n    import torch\\nexcept ImportError:   # local smoke without torch\\n    torch = None\\nfrom scipy.cluster.hierarchy import fcluster, linkage\\nfrom scipy.sparse import csr_matrix, diags, hstack, vstack\\nfrom scipy.spatial.distance import squareform\\nfrom sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.svm import LinearSVC\\n\\nSMOKE = os.environ.get(\\\'NTL_SMOKE\\\') == \\\'1\\\'   # local code-path check: pseudo features instead of checkpoints, CPU\\nT0 = time.time()\\nLABELS = sorted([\\\'gpt-5.5\\\', \\\'gpt-5.6-luna\\\', \\\'gpt-5.6-sol\\\', \\\'gpt-5.6-terra\\\', \\\'gpt-6-astra\\\',\\n                 \\\'gemini-3.1-pro-preview\\\', \\\'gemini-3.6-flash\\\', \\\'gemini-3.7-flash\\\', \\\'gemini-3.8-flash\\\',\\n                 \\\'claude-fable-5\\\', \\\'claude-opus-4-6\\\', \\\'claude-opus-4-8\\\', \\\'claude-opus-5\\\', \\\'claude-sonnet-5\\\',\\n                 \\\'qwen3.8-max\\\', \\\'glm-5.3\\\', \\\'glm-5.3-flash\\\', \\\'kimi-k3\\\'])\\nK = len(LABELS)\\nLAB2I = {l: i for i, l in enumerate(LABELS)}\\nFAMILY_IDX = {f: np.array([i for i, l in enumerate(LABELS) if l.startswith(f)]) for f in (\\\'claude\\\', \\\'gpt\\\', \\\'gemini\\\', \\\'glm\\\')}\\n\\n# ---- test-pool grouping\\nGROUP_THRESH, GROUP_LAYER, GROUP_MIN, GROUP_MAX = 0.60, 24, 2, 18   # mixed content + GLM-L24 distance, average linkage\\nSTYLE_GROUP_THRESH = 0.90                                           # stylegram\\\'s own median-normalised content distance\\n\\n# ---- blend weights: log-probability multipliers of the 0.563 notebook (its L-BFGS fit on prompt-grouped OOF)\\nW = {\\\'stylegram\\\': 1.516, \\\'stylometry_lgb\\\': 0.063, \\\'ultrabert\\\': 0.095,\\n     \\\'q3_8b\\\': 0.0, \\\'q25_7b\\\': 0.0, \\\'q3_4b\\\': 0.0, \\\'glm9b\\\': 0.0, \\\'debv3l\\\': 0.05, \\\'electl\\\': 0.032, \\\'desk\\\': 0.132,\\n     \\\'lora24_public\\\': 0.177, \\\'lora24_seed7\\\': 0.113, \\\'lora24_claude\\\': 0.177, \\\'lora20_q25_instyle\\\': 0.17,\\n     \\\'tokattn_q35\\\': 0.153, \\\'tokattn_deb\\\': 0.185}\\nTWIN_W = {\\\'q3_8b\\\': 1.5 * 0.219, \\\'q25_7b\\\': 1.5 * 0.141, \\\'q3_4b\\\': 1.5 * 0.083, \\\'glm9b\\\': 1.5 * 0.110, \\\'debv3l\\\': 0.25}\\nLORA_TWIN_W = {\\\'lora24_public\\\': 0.15, \\\'lora24_claude\\\': 0.15}\\nGATE_LAM = {\\\'claude_style\\\': 1.5, \\\'claude_lora\\\': 1.0, \\\'family\\\': 0.25}\\nBALANCE_ALPHA = 0.5\\nEXTRA_MIN_LABELS, EXTRA_SHARE = 12, 0.5\\n\\n# ---- the addition: shared-prompt Gaussian group heads on four measured views, weight = the centred twin of the same backbone,\\n#      reduction / temperatures from our production fit (research/gaussian/production_weights_v8g.json, 135 training prompts)\\nGAUSS_HEADS = {\\\'gauss_q3_8b_L22\\\': dict(src=\\\'q3_8b\\\', part=\\\'L22_mean\\\', d=384, T1=2.351, Tg=2.306, w=TWIN_W[\\\'q3_8b\\\']),\\n               \\\'gauss_glm9b_L8\\\': dict(src=\\\'glm9b\\\', part=\\\'L8_mean\\\', d=384, T1=2.722, Tg=2.596, w=TWIN_W[\\\'glm9b\\\']),\\n               \\\'gauss_q3_4b_L4\\\': dict(src=\\\'q3_4b\\\', part=\\\'L4_mean\\\', d=384, T1=2.725, Tg=2.578, w=TWIN_W[\\\'q3_4b\\\']),\\n               \\\'gauss_debv3l_L24\\\': dict(src=\\\'debv3l\\\', part=\\\'L24_mean\\\', d=384, T1=3.112, Tg=3.069, w=TWIN_W[\\\'debv3l\\\'])}\\nGAUSS_SETTING = dict(red=\\\'wpca\\\', se_shrink=\\\'lw\\\', sp_shrink=\\\'lw\\\', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)\\n\\n# ---- member specifications\\nLM_HEADS = [  # key, model_type, hidden size, parts of the plain head [(layer, pool)], extra parts extracted in the same pass, C\\n    (\\\'glm9b\\\', \\\'glm4\\\', 4096, [(8, \\\'mean\\\'), (3, \\\'mean\\\')], [(GROUP_LAYER, \\\'mean\\\')], 0.0025),   # first: its L24 builds the groups\\n    (\\\'q3_8b\\\', \\\'qwen3\\\', 4096, [(7, \\\'mean\\\'), (22, \\\'mean\\\')], [], 0.0025),\\n    (\\\'q25_7b\\\', \\\'qwen2\\\', 3584, [(6, \\\'mean\\\'), (20, \\\'mean\\\')], [], 0.0025),\\n    (\\\'q3_4b\\\', \\\'qwen3\\\', 2560, [(4, \\\'style\\\'), (7, \\\'style\\\')], [(4, \\\'mean\\\')], 0.0025)]\\nENC_HEADS = [  # key, input directory name, parts, C\\n    (\\\'debv3l\\\', \\\'deberta-v3-large\\\', [(24, \\\'mean\\\'), (6, \\\'mean\\\')], 0.005),\\n    (\\\'electl\\\', \\\'electra-large-discriminator\\\', [(21, \\\'mean\\\'), (24, \\\'style\\\')], 0.005),\\n    (\\\'desk\\\', \\\'desklib-detector\\\', [(23, \\\'mean\\\'), (23, \\\'style\\\')], 0.005)]\\nLORA_RUNS = [  # (trunk model_type, hidden, kept layers), pooled layers, [(key, checkpoint marker, norms file, anchor features file)]\\n    ((\\\'qwen3\\\', 4096, 24), [2, 4, 6, 8, 12, 16, 20, 24], [\\n        (\\\'lora24_public\\\', \\\'**/lora24/final/COMPLETE.json\\\', \\\'heads.safetensors\\\', \\\'anchor_features.npz\\\'),\\n        (\\\'lora24_seed7\\\', \\\'**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json\\\', \\\'norms.safetensors\\\', \\\'valid_features.npz\\\'),\\n        (\\\'lora24_claude\\\', \\\'**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json\\\', \\\'heads.safetensors\\\', \\\'anchor_features.npz\\\')]),\\n    ((\\\'qwen2\\\', 3584, 20), [2, 4, 6, 8, 10, 12, 16, 20], [\\n        (\\\'lora20_q25_instyle\\\', \\\'**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json\\\', \\\'norms.safetensors\\\', \\\'valid_features.npz\\\')])]\\nLORA_MAX_LEN, LORA_BS = 447, 8\\nTOKATTN_HEADS = [(\\\'tokattn_q35\\\', \\\'tokattn_final.pt\\\', \\\'qwen3_5\\\'), (\\\'tokattn_deb\\\', \\\'tokattn_debl23_final.pt\\\', \\\'deberta\\\')]\\n\\n# ---- data\\nif SMOKE:\\n    _all = pd.read_csv(os.environ[\\\'NTL_DATA\\\'] + \\\'/train.csv\\\')\\n    _p = _all[\\\'prompt_id\\\'].drop_duplicates().sample(frac=1, random_state=0).tolist()\\n    train = _all[_all[\\\'prompt_id\\\'].isin(_p[:30])].reset_index(drop=True)\\n    test = _all[_all[\\\'prompt_id\\\'].isin(_p[30:42])].sample(frac=1, random_state=1).reset_index(drop=True)\\n    test = pd.concat([test[test[\\\'prompt_id\\\'] == p].head(k) for p, k in zip(_p[30:42], [1, 1, 2, 2, 2, 3, 3, 4, 6, 9, 18, 18])]).reset_index(drop=True)\\n    if os.environ.get(\\\'NTL_TINY\\\') == \\\'1\\\':\\n        test = test.head(3).reset_index(drop=True)\\n    SMOKE_Y = test[\\\'model\\\'].map(LAB2I).to_numpy(dtype=int)\\nelse:\\n    DATA_DIR = next(p for p in [pathlib.Path(\\\'/kaggle/input/competitions/name-that-llm\\\'), pathlib.Path(\\\'/kaggle/input/name-that-llm\\\')]\\n                    if (p / \\\'train.csv\\\').exists())\\n    train, test = pd.read_csv(DATA_DIR / \\\'train.csv\\\'), pd.read_csv(DATA_DIR / \\\'test.csv\\\')\\ntest[\\\'text\\\'] = test[\\\'text\\\'].fillna(\\\'\\\').astype(str)\\nTR, TE = train[\\\'text\\\'].tolist(), test[\\\'text\\\'].tolist()\\ny = train[\\\'model\\\'].map(LAB2I).to_numpy(dtype=int)\\nPROMPT = train[\\\'prompt_id\\\'].to_numpy(dtype=object)\\nTINY = len(TE) < 50   # the interactive 3-row test: exercise the group paths on training rows so the hidden rerun cannot hit untested code\\nDEV = \\\'cuda\\\' if torch is not None and torch.cuda.is_available() else \\\'cpu\\\'\\nprint(f\\\'train {train.shape}  test {test.shape}  gpus {torch.cuda.device_count() if torch is not None else 0}  smoke {SMOKE}\\\', flush=True)\\n\\nLOGIT = np.zeros((len(TE), K))           # the running log-probability blend\\n\\n\\ndef add_member(key, prob, rows=None, weight=None):\\n    """Add weight * log(prob) to the blend (all rows, or the given row subset)."""\\n    w = W[key] if weight is None else weight\\n    lp = np.log(np.clip(prob, 1e-9, 1.0))\\n    if rows is None:\\n        LOGIT[:] += w * lp\\n    else:\\n        LOGIT[rows] += w * lp\\n    print(f\\\'  + {key:20s} weight {w:.3f} on {len(TE) if rows is None else int(np.sum(rows))} rows  ({time.time() - T0:.0f}s)\\\', flush=True)\\n\\n\\ndef softmax_rows(S):\\n    S = S - S.max(1, keepdims=True)\\n    P = np.exp(S)\\n    return P / P.sum(1, keepdims=True)\\n\\n\\ndef full_prob(P_part, classes):\\n    """Expand a (n, len(classes)) probability block to (n, K) on the global label order."""\\n    P = np.zeros((len(P_part), K))\\n    P[:, classes] = P_part\\n    return P\\n\\n\\ndef group_centre_dense(X, groups):\\n    X = np.asarray(X, dtype=np.float32).copy()\\n    for g in np.unique(groups):\\n        i = groups == g\\n        X[i] -= X[i].mean(0)\\n    return X\\n\\n\\ndef self_test_rows():\\n    """Training rows of the first three prompts, grouped by their true prompt (used only when the test file is tiny)."""\\n    idx = np.where(np.isin(PROMPT, pd.unique(PROMPT)[:3]))[0]\\n    return idx, np.unique(PROMPT[idx], return_inverse=True)[1]\\n\\n# ============================================================ 1. grouping the test pool\\ndef average_linkage_groups(D, thresh):\\n    """Cluster ids (1-based) from a square distance matrix, average linkage, cut at thresh."""\\n    if len(D) < 2:\\n        return np.ones(len(D), dtype=int)\\n    D = np.asarray(D, dtype=np.float64).copy()\\n    np.fill_diagonal(D, 0.0)\\n    return fcluster(linkage(squareform(D, checks=False), method=\\\'average\\\'), thresh, criterion=\\\'distance\\\')\\n\\n\\ndef content_cosine_distance(texts, max_features=None):\\n    X = TfidfVectorizer(stop_words=\\\'english\\\', sublinear_tf=True, max_features=max_features, dtype=np.float32).fit_transform(texts)\\n    return np.clip(1.0 - (X @ X.T).toarray(), 0.0, 1.0)\\n\\n\\ndef mixed_groups(texts, E, thresh=GROUP_THRESH):\\n    """Average of the content-word TF-IDF cosine distance and the cosine distance of mean-centred, L2-normalised embeddings E."""\\n    if len(texts) < 2:\\n        return np.ones(len(texts), dtype=int)\\n    Z = np.asarray(E, dtype=np.float64)\\n    Z = Z - Z.mean(0)\\n    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12\\n    D = 0.5 * (content_cosine_distance(texts) + np.clip(1.0 - Z @ Z.T, 0.0, 2.0))\\n    return average_linkage_groups(D, thresh)\\n\\n\\ndef normalised_content_groups(texts, thresh=STYLE_GROUP_THRESH):\\n    """The stylegram\\\'s grouping: content distance divided by its median off-diagonal value, cut at thresh."""\\n    if len(texts) < 2:\\n        return np.ones(len(texts), dtype=int)\\n    D = content_cosine_distance(texts, max_features=100_000)\\n    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)\\n    return average_linkage_groups(D / med, thresh)\\n\\n\\ndef eligible(groups, lo=GROUP_MIN, hi=GROUP_MAX):\\n    size = np.bincount(groups)[groups]\\n    return (size >= lo) & (size <= hi)\\n\\n\\ndef describe_groups(name, groups):\\n    size = np.bincount(groups)[groups]\\n    print(f\\\'{name}: {len(groups)} rows | singletons {int((size == 1).sum())}  2-3: {int(((size >= 2) & (size <= 3)).sum())}\\\'\\n          f\\\'  4-18: {int(((size >= 4) & (size <= 18)).sum())}  >18: {int((size > 18).sum())}\\\', flush=True)\\n\\n\\nG_TE, GROUP_ROWS = None, None   # set after the GLM pass (section 4)\\n\\n# ============================================================ 2. style n-grams + extra passages + gates\\nFUNCTION_WORDS = frozenset("a an the and or but if while although because as so than that which who whom whose this these those it its "\\n                           "they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with "\\n                           "without for of by over under between through into about is are was were be been being have has had do does did "\\n                           "can could will would shall should may might must not no nor all any some each every both either neither other "\\n                           "another such only also just very too more most less least here there when where why how what whether then now "\\n                           "still yet even much many few little".split())\\nTOKEN_RE = re.compile(r"\\\\w+|[^\\\\w\\\\s]")\\n\\n\\ndef shape_view(text):\\n    return re.sub(r"[^\\\\W\\\\d_]+", lambda m: \\\'X\\\' if m[0].isupper() else \\\'x\\\', text)\\n\\n\\ndef function_view(text):\\n    return \\\' \\\'.join(t.lower() if (t.lower() in FUNCTION_WORDS or not t.isalnum()) else \\\'CONTENT\\\' for t in TOKEN_RE.findall(text))\\n\\n\\nclass StyleGrams:\\n    """Four stacked TF-IDF views: char 3-5, word 1-3, shape (char 2-6 of the x/X text), function-word (word 1-3 of the masked text)."""\\n    VIEWS = [(\\\'char\\\', str, (3, 5), 250_000, 1.0), (\\\'word\\\', str, (1, 3), 180_000, 0.7),\\n             (\\\'char\\\', shape_view, (2, 6), 60_000, 0.5), (\\\'word\\\', function_view, (1, 3), 50_000, 0.5)]\\n\\n    def fit(self, texts):\\n        self.vecs = []\\n        for analyzer, view, ngr, maxf, w in self.VIEWS:\\n            opts = dict(analyzer=analyzer, ngram_range=ngr, lowercase=False, sublinear_tf=True, min_df=2, max_features=maxf, dtype=np.float64)\\n            if analyzer == \\\'word\\\':\\n                opts[\\\'token_pattern\\\'] = r"(?u)\\\\b\\\\w+\\\\b|[^\\\\w\\\\s]"\\n            self.vecs.append((TfidfVectorizer(**opts).fit([view(t) for t in texts]), view, w))\\n        return self\\n\\n    def transform(self, texts):\\n        return hstack([v.transform([view(t) for t in texts]) * w for v, view, w in self.vecs], format=\\\'csr\\\')\\n\\n\\ndef sparse_group_centre(X, groups, lo=GROUP_MIN, hi=GROUP_MAX):\\n    """Subtract each group\\\'s mean row (sparse); groups outside [lo, hi] are left as they are."""\\n    X = csr_matrix(X)\\n    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)\\n    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))\\n    if not len(keep):\\n        return X.copy()\\n    col = np.full(len(cnt), -1); col[keep] = np.arange(len(keep))\\n    rows = np.flatnonzero(col[inv] >= 0)\\n    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))\\n    means = diags(1.0 / cnt[keep]) @ (M.T @ X)\\n    return (X - M @ means).tocsr()\\n\\n\\ndef svc_probs(svc, X):\\n    return full_prob(softmax_rows(svc.decision_function(X)), svc.classes_)\\n\\n\\n# ---- public in-style passages (prompt-bank protocol), one per (prompt, label); prompts answered by >= EXTRA_MIN_LABELS labels\\ndef load_extra_passages():\\n    sources = [(\\\'**/ntl_extra_passages_claude.csv\\\', \\\'model\\\', None), (\\\'**/gen_passages.csv\\\', \\\'model\\\', None),\\n               (\\\'**/ntl_generations.csv\\\', \\\'label\\\', \\\'finish_reason\\\'), (\\\'**/ntl_proxy_generations.csv\\\', \\\'label\\\', None),\\n               (\\\'**/ntl_generated*.csv\\\', \\\'model\\\', \\\'finish\\\'), (\\\'**/passages_clean.csv\\\', \\\'model\\\', None)]\\n    parts = []\\n    for pattern, label_col, finish_col in sources:\\n        for f in sorted(glob.glob(\\\'/kaggle/input/\\\' + pattern, recursive=True)):\\n            d = pd.read_csv(f)\\n            if finish_col and finish_col in d:\\n                d = d[d[finish_col].eq(\\\'stop\\\')]\\n            d = d.rename(columns={label_col: \\\'model\\\'})[[\\\'prompt_id\\\', \\\'model\\\', \\\'text\\\']].copy()\\n            d[\\\'prompt_id\\\'] = (\\\'pv_\\\' if f.endswith(\\\'passages_clean.csv\\\') else \\\'\\\') + d[\\\'prompt_id\\\'].astype(str)   # prvsiyan\\\'s prompts are a different bank\\n            parts.append(d)\\n    if not parts:\\n        return pd.DataFrame(columns=[\\\'prompt_id\\\', \\\'model\\\', \\\'text\\\'])\\n    x = pd.concat(parts, ignore_index=True)\\n    x = x[x[\\\'model\\\'].isin(LABELS)].copy()\\n    x[\\\'text\\\'] = x[\\\'text\\\'].astype(str).map(lambda t: re.sub(r\\\'\\\\s+\\\', \\\' \\\', re.sub(r\\\'^\\\\*\\\\*(.*)\\\\*\\\\*$\\\', r\\\'\\\\1\\\', t.strip())).strip())\\n    x = x[x[\\\'text\\\'].str.split().str.len().between(30, 300)].drop_duplicates(subset=[\\\'model\\\', \\\'text\\\'])\\n    x = x.drop_duplicates(subset=[\\\'prompt_id\\\', \\\'model\\\'])\\n    x = x[x.groupby(\\\'prompt_id\\\')[\\\'model\\\'].transform(\\\'nunique\\\') >= EXTRA_MIN_LABELS].reset_index(drop=True)\\n    return x\\n\\n\\nEXTRA = load_extra_passages()\\nEX_T, EX_Y, EX_G = EXTRA[\\\'text\\\'].tolist(), EXTRA[\\\'model\\\'].map(LAB2I).to_numpy(dtype=int), EXTRA[\\\'prompt_id\\\'].to_numpy(dtype=object)\\nprint(f\\\'extra passages: {len(EXTRA)} rows, {EXTRA["prompt_id"].nunique()} prompts\\\', flush=True)\\n\\n\\ndef extra_sample_weights():\\n    """Per class: original rows keep (1 - EXTRA_SHARE) of the class weight, the class\\\'s extra rows share EXTRA_SHARE of it."""\\n    n_tr, n_ex = np.bincount(y, minlength=K).astype(float), np.bincount(EX_Y, minlength=K).astype(float)\\n    w_tr = np.where(n_ex[y] > 0, 1.0 - EXTRA_SHARE, 1.0)\\n    w_ex = EXTRA_SHARE * n_tr[EX_Y] / np.maximum(n_ex[EX_Y], 1.0)\\n    return np.concatenate([w_tr, w_ex])\\n\\n\\ngrams = StyleGrams().fit(TR + EX_T)                       # vocabulary and idf from train + extra\\nX_tr, X_ex, X_te = grams.transform(TR), grams.transform(EX_T) if len(EX_T) else None, grams.transform(TE)\\n\\n\\ndef fit_svc(X, labels, C=0.5, **kw):\\n    return LinearSVC(C=C, random_state=42, max_iter=4000).fit(X, labels, **kw)\\n\\n\\nif X_ex is not None:\\n    svc_plain = fit_svc(vstack([X_tr, X_ex]).tocsr(), np.concatenate([y, EX_Y]), sample_weight=extra_sample_weights())\\nelse:\\n    svc_plain = fit_svc(X_tr, y)\\nP_style = svc_probs(svc_plain, X_te)\\n\\n# centred twin: prompt-centred training rows (extra rows centred within their own prompts) -> inferred test groups\\nX_tr_c = sparse_group_centre(X_tr, PROMPT)\\nif X_ex is not None:\\n    svc_centred = fit_svc(vstack([X_tr_c, sparse_group_centre(X_ex, EX_G, GROUP_MIN, 10 ** 6)]).tocsr(), np.concatenate([y, EX_Y]),\\n                          sample_weight=extra_sample_weights())\\nelse:\\n    svc_centred = fit_svc(X_tr_c, y)\\nG_style = normalised_content_groups(TE)\\nrows_style = eligible(G_style)\\ndescribe_groups(\\\'stylegram groups\\\', G_style)\\nif rows_style.any():\\n    P_style[rows_style] = svc_probs(svc_centred, sparse_group_centre(X_te[rows_style], G_style[rows_style]))\\nif TINY:   # exercise the centred path on training rows grouped by their true prompt (output discarded)\\n    st_idx, st_g = self_test_rows()\\n    acc = np.mean(svc_centred.classes_[svc_centred.decision_function(sparse_group_centre(X_tr[st_idx], st_g)).argmax(1)] == y[st_idx])\\n    print(f\\\'  centred stylegram self-test on {len(st_idx)} training rows (seen in training): acc {acc:.2f}\\\', flush=True)\\nadd_member(\\\'stylegram\\\', P_style)\\ndel X_tr_c, svc_plain, svc_centred, P_style\\n\\n\\n# ---- gates: style SVCs that only re-rank inside one family after the blend\\ndef family_gate(texts, labels, classes, C, sample_weight=None, balanced=False):\\n    g = StyleGrams().fit(texts)\\n    svc = LinearSVC(C=C, class_weight=\\\'balanced\\\' if balanced else None, random_state=42, max_iter=4000).fit(\\n        g.transform(texts), labels, sample_weight=sample_weight)\\n    assert list(svc.classes_) == list(classes)\\n    d = svc.decision_function(g.transform(TE))\\n    return np.column_stack([-d, d]) if len(classes) == 2 else d\\n\\n\\nGATES = {}\\napi_claude = sorted(glob.glob(\\\'/kaggle/input/**/ntl_extra_passages_claude.csv\\\', recursive=True))\\nif api_claude:\\n    api = pd.read_csv(api_claude[0])\\n    m = np.isin(y, FAMILY_IDX[\\\'claude\\\'])\\n    GATES[\\\'claude\\\'] = family_gate([t for t, k in zip(TR, m) if k] + api[\\\'text\\\'].astype(str).tolist(),\\n                                  np.concatenate([y[m], api[\\\'model\\\'].map(LAB2I).to_numpy(dtype=int)]), FAMILY_IDX[\\\'claude\\\'], 0.5)\\n    print(f\\\'claude gate: {int(m.sum())} official + {len(api)} API rows\\\', flush=True)\\n    del api\\nelse:\\n    print(\\\'Claude API passages not attached: Claude gate off\\\', flush=True)\\nfor fam, C in [(\\\'gpt\\\', 1.0), (\\\'gemini\\\', 1.0), (\\\'glm\\\', 0.5)]:\\n    m, me = np.isin(y, FAMILY_IDX[fam]), np.isin(EX_Y, FAMILY_IDX[fam])\\n    GATES[fam] = family_gate([t for t, k in zip(TR, m) if k] + [t for t, k in zip(EX_T, me) if k], np.concatenate([y[m], EX_Y[me]]),\\n                             FAMILY_IDX[fam], C, sample_weight=np.concatenate([np.ones(int(m.sum())), np.full(int(me.sum()), 0.25)]), balanced=True)\\n    print(f\\\'{fam} gate: {int(m.sum())} official + {int(me.sum())} extra rows\\\', flush=True)\\ndel grams, X_tr, X_ex, X_te\\n\\n# ============================================================ 3. stylometric features + LightGBM\\nPUNCT = list(\\\',.;:!?()[]"\\\\\\\'-\\\') + [\\\'—\\\', \\\'–\\\', \\\'‘\\\', \\\'’\\\', \\\'“\\\', \\\'”\\\', \\\'…\\\', \\\'*\\\', \\\'/\\\', \\\'%\\\']\\nSTOPS = sorted(ENGLISH_STOP_WORDS)\\nSENT_RE, WORD_RE = re.compile(r\\\'(?<=[.!?])\\\\s+\\\'), re.compile(r"[A-Za-z’\\\']+")\\n\\n\\ndef stylometry(t):\\n    words = WORD_RE.findall(t)\\n    low = [w.lower() for w in words]\\n    nw, nc = max(len(words), 1), max(len(t), 1)\\n    sents = [s for s in SENT_RE.split(t.strip()) if s]\\n    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], dtype=float)\\n    wl = np.array([len(w) for w in words] or [0], dtype=float)\\n    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw,\\n         sum(c.isupper() for c in t) / nc, sum(c.isdigit() for c in t) / nc, sum(ord(c) > 127 for c in t) / nc,\\n         t.count(\\\', \\\') / nw, t.count(\\\' and \\\') / nw, t.count(\\\' but \\\') / nw,\\n         sum(1 for s in sents if s[:1].isupper() and s.split()[0].lower() in (\\\'this\\\', \\\'these\\\', \\\'it\\\', \\\'they\\\')) / max(len(sents), 1),\\n         sum(w.endswith(\\\'ly\\\') for w in words) / nw, sum(w.endswith(\\\'ing\\\') for w in words) / nw, sum(w.endswith(\\\'tion\\\') for w in words) / nw]\\n    f += [t.count(q) / nw for q in PUNCT]\\n    counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)\\n    f += [counts.get(s, 0) / nw for s in STOPS]\\n    return f\\n\\n\\nF_tr, F_te = (np.array([stylometry(t) for t in txt], dtype=np.float32) for txt in (TR, TE))\\ntry:\\n    import lightgbm as lgb\\n    booster = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.8, subsample_freq=1,\\n                                 colsample_bytree=0.5, reg_lambda=1.0, random_state=0, verbose=-1, n_jobs=4).fit(F_tr, y)\\n    add_member(\\\'stylometry_lgb\\\', full_prob(booster.predict_proba(F_te), booster.classes_))\\n    del booster\\nexcept ImportError:\\n    assert SMOKE, \\\'lightgbm missing\\\'\\n    print(\\\'smoke: lightgbm unavailable, stylometry member skipped\\\', flush=True)\\ndel F_tr, F_te\\n\\n# ============================================================ 4b. shared-prompt Gaussian group head\\n# Verbatim core of our research tool tools/group_gaussian.py (sha256 8a42427b77a37f56...), numpy only.\\n# Model per dense view, in a reduced space z (standardise -> within-prompt-residual PCA, fitted on training rows):\\n#   z_i = mu_{y_i} + p_g + e_i,  p_g ~ N(0, S_p) shared by every passage of prompt group g,  e_i ~ N(0, S_e).\\n# A joint diagonalisation makes the dimensions independent; for a detected group of n passages the exact joint\\n# log-likelihood of a label tuple is sum_i a_i(c_i) + sum_{i<j} b_ij(c_i, c_j). Groups of 2-3 are decoded exactly over\\n# all label tuples with the at-most-once constraint; larger groups by a cavity EM with the finite-population-corrected\\n# prompt-effect estimate lam / (1 + (n-1) lam); singletons integrate the prompt effect out (never used here).\\nimport itertools\\nK = 18\\nDEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,\\n           "iters": 5, "repel": True, "sp_scale": 1.0}\\n\\n# =================================================================== primitives ====\\ndef _lse(A, axis=None, keepdims=False):\\n    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)\\n    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m\\n    if keepdims: return out\\n    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())\\n\\ndef log_softmax(L): return L - _lse(L, axis=1, keepdims=True)\\n\\ndef _lw_shrinkage(X):\\n    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""\\n    from sklearn.covariance import ledoit_wolf_shrinkage\\n    return float(ledoit_wolf_shrinkage(X, assume_centered=True))\\n\\ndef _shrunk(S, a):\\n    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)\\n\\ndef _two_way(Z, y, pinv, G, iters=4):\\n    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""\\n    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)\\n    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))\\n    for _ in range(iters):\\n        R = Z - P[pinv]\\n        for c in cls: mu[c] = R[y == c].mean(0)\\n        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]\\n    return mu, P, cnt_p\\n\\ndef fit(X, y, prompts, cfg=None):\\n    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.\\n    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller\\\'s cross-fit)."""\\n    c = dict(DEFAULT); c.update(cfg or {})\\n    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape\\n    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)\\n    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx\\n    if c["red"] == "none":\\n        V = np.eye(D)\\n    else:\\n        if c["red"] == "wpca":\\n            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]\\n        elif c["red"] == "pca": R = Xs\\n        else: raise ValueError(c["red"])\\n        d = int(min(c["d"], N - G - 1, D))\\n        # economy SVD through the (N x N) Gram matrix (N << D for the real views)\\n        if N < D:\\n            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]\\n            V = (R.T @ U) / np.sqrt(w)\\n        else:\\n            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T\\n    Z = Xs @ V; d = Z.shape[1]\\n    mu, P, ng = _two_way(Z, y, pinv, G)\\n    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)\\n    Se_raw = E.T @ E / dof\\n    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)\\n    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw\\n    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])\\n    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]\\n    # joint diagonalisation: A Se A\\\' = I, A Sp A\\\' = diag(lam)\\n    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)\\n    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li\\n    W = V @ A.T\\n    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,\\n            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}\\n\\ndef transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]\\n\\n_MASK = {}\\ndef _inj_mask(n, C):\\n    if (n, C) not in _MASK:\\n        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)\\n        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]\\n        _MASK[(n, C)] = bad\\n    return _MASK[(n, C)]\\n\\ndef singleton_ll(U, m, lam):\\n    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""\\n    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)\\n\\ndef joint_terms(U, m, lam):\\n    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""\\n    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]\\n    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)\\n    return a, B\\n\\ndef joint_table(a, B):\\n    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""\\n    n, C = a.shape; Q = np.zeros([C] * n)\\n    for i in range(n):\\n        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)\\n    for i, j in itertools.combinations(range(n), 2):\\n        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)\\n    return Q\\n\\ndef exact_marginals(Q, T=1.0, inj=True):\\n    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""\\n    n = Q.ndim; C = Q.shape[0]; Qt = Q / T\\n    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)\\n    lz = _lse(Qt); out = np.empty((n, C))\\n    for i in range(n):\\n        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz\\n    return out\\n\\ndef em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):\\n    """cavity EM for one group (any n >= 2): returns (n, C) log q."""\\n    n = len(U); v = lam / (1 + (n - 1) * lam)\\n    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))\\n    q = np.exp(log_softmax(L))\\n    for _ in range(iters):\\n        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)\\n        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T\\n        if repel:   # mean-field at-most-once: discourage classes the other members already hold\\n            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)\\n        q = np.exp(log_softmax(L))\\n    return log_softmax(L)\\n\\nLAM_DIFF = 1e6\\ndef lam_g(prm, cfg=None):\\n    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg \\\'diff\\\'): then the joint likelihood\\n    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs\\n    centred prototypes) and the prompt term cancels exactly."""\\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})\\n    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]\\n\\ndef group_logpost(U, prm, T1=None, Tg=None, cfg=None):\\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)\\n    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg\\n    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)\\n    if n <= c["n_exact"]:\\n        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])\\n    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])\\n\\ndef scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):\\n    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""\\n    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))\\n    sizes = np.bincount(lab); s1 = sizes[lab] == 1\\n    T1 = prm["T1"] if T1 is None else T1\\n    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)\\n    for g in np.where(sizes >= 2)[0]:\\n        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)\\n    return out\\n\\n# ============================================================ 4. frozen models: plain heads, centred twins, Gaussian group heads\\nif not SMOKE:\\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\\nCACHED_PARTS = {}   # key -> {part name: (train features, test features)}\\n\\n\\ndef find_checkpoint(model_type, hidden=None, dir_name=None):\\n    """A checkpoint directory under /kaggle/input by model_type (+ hidden size) or by directory name."""\\n    pattern = f\\\'/kaggle/input/**/{dir_name}/config.json\\\' if dir_name else \\\'/kaggle/input/**/config.json\\\'\\n    for cfg in sorted(glob.glob(pattern, recursive=True)):\\n        try:\\n            c = json.load(open(cfg))\\n        except Exception:\\n            continue\\n        if dir_name or (c.get(\\\'model_type\\\') == model_type and (hidden is None or c.get(\\\'hidden_size\\\') == hidden)):\\n            directory = os.path.dirname(cfg)\\n            if _ntl_has_model_weights(directory):\\n                return directory\\n    return None\\n\\n\\ndef load_truncated_decoder(path, keep, dtype):\\n    cfg = AutoConfig.from_pretrained(path)\\n    cfg.num_hidden_layers = keep\\n    if getattr(cfg, \\\'layer_types\\\', None):\\n        cfg.layer_types = cfg.layer_types[:keep]\\n    kw = dict(config=cfg, dtype=dtype)\\n    if DEV == \\\'cuda\\\':\\n        kw[\\\'device_map\\\'] = \\\'auto\\\'\\n    return AutoModel.from_pretrained(path, **kw).eval()\\n\\n\\ndef style_token_mask(tok):\\n    """Vocabulary mask of \\\'style\\\' tokens: the decoded piece is an English stop word or pure punctuation."""\\n    pieces = [tok.decode([i]) for i in range(len(tok))]\\n    return torch.tensor([(p.strip().lower() in ENGLISH_STOP_WORDS) or bool(re.fullmatch(r\\\'\\\\s*[^\\\\w\\\\s]+\\\\s*\\\', p)) for p in pieces])\\n\\n\\n@torch.no_grad()\\ndef pooled_states(model, tok, texts, parts, decoder, bs, max_len=512):\\n    """{\\\'L{layer}_{pool}\\\': (n, hidden)} masked mean pools of hidden_states[layer] (all tokens, or style tokens only)."""\\n    H = model.config.hidden_size\\n    out = {f\\\'L{L}_{pool}\\\': np.zeros((len(texts), H), dtype=np.float32) for L, pool in parts}\\n    smask = style_token_mask(tok) if any(pool == \\\'style\\\' for _, pool in parts) else None\\n    dev = next(model.parameters()).device\\n    order = np.argsort([len(t) for t in texts])\\n    retried = 0\\n    for b0 in range(0, len(texts), bs):\\n        idx = order[b0:b0 + bs]\\n        if decoder:\\n            batch = [(tok.bos_token or \\\'\\\') + texts[i] for i in idx]\\n            enc = tok(batch, return_tensors=\\\'pt\\\', padding=True, truncation=True, max_length=max_len, add_special_tokens=False).to(dev)\\n        else:\\n            enc = tok([texts[i] for i in idx], return_tensors=\\\'pt\\\', padding=True, truncation=True, max_length=max_len).to(dev)\\n        hs = model(**enc, output_hidden_states=True).hidden_states\\n        if decoder and not all(torch.isfinite(hs[L]).all() for L, _ in parts):   # fp16 overflow: redo the batch in bf16\\n            retried += 1\\n            model.to(torch.bfloat16)\\n            hs = model(**enc, output_hidden_states=True).hidden_states\\n        m_all = enc[\\\'attention_mask\\\'].float()\\n        for L, pool in parts:\\n            m = m_all * smask.to(m_all.device)[enc[\\\'input_ids\\\']].float() if pool == \\\'style\\\' else m_all\\n            m = m.unsqueeze(-1).to(hs[L].device)\\n            out[f\\\'L{L}_{pool}\\\'][idx] = ((hs[L].float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()\\n        del hs, enc\\n    if retried:\\n        print(f\\\'  {retried} batches redone in bf16\\\', flush=True)\\n    return {k: np.nan_to_num(v) for k, v in out.items()}\\n\\n\\ndef pseudo_parts(parts, dim=48):\\n    """SMOKE only: deterministic stand-ins for the model passes (TF-IDF SVD of the texts), same downstream code."""\\n    from sklearn.decomposition import TruncatedSVD\\n    tf = TfidfVectorizer(analyzer=\\\'char\\\', ngram_range=(2, 4), sublinear_tf=True, min_df=2).fit(TR + TE)\\n    out = {}\\n    for j, (L, pool) in enumerate(parts):\\n        svd = TruncatedSVD(dim, random_state=L + 7 * j).fit(tf.transform(TR))\\n        out[f\\\'L{L}_{pool}\\\'] = (svd.transform(tf.transform(TR)).astype(np.float32), svd.transform(tf.transform(TE)).astype(np.float32))\\n    return out\\n\\n\\ndef extract_parts(key, parts, loader):\\n    """Train and test pools for every requested part; loader() returns (model, tok, decoder flag, batch size)."""\\n    if SMOKE:\\n        return pseudo_parts(parts)\\n    model, tok, decoder, bs = loader()\\n    tr, te = pooled_states(model, tok, TR, parts, decoder, bs), pooled_states(model, tok, TE, parts, decoder, bs)\\n    del model, tok\\n    gc.collect()\\n    if DEV == \\\'cuda\\\':\\n        torch.cuda.empty_cache()\\n    return {k: (tr[k], te[k]) for k in tr}\\n\\n\\ndef decoder_loader(model_type, hidden, keep):\\n    def load():\\n        path = find_checkpoint(model_type, hidden)\\n        assert path, f\\\'no {model_type} checkpoint with hidden size {hidden} under /kaggle/input\\\'\\n        tok = AutoTokenizer.from_pretrained(path)\\n        tok.padding_side = \\\'right\\\'\\n        if tok.pad_token is None:\\n            tok.pad_token = tok.eos_token\\n        return load_truncated_decoder(path, keep, torch.float16 if DEV == \\\'cuda\\\' else torch.float32), tok, True, 8\\n    return load\\n\\n\\ndef encoder_loader(dir_name):\\n    def load():\\n        path = find_checkpoint(None, dir_name=dir_name)\\n        assert path, f\\\'encoder directory {dir_name} not found under /kaggle/input\\\'\\n        return AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval(), AutoTokenizer.from_pretrained(path), False, 16\\n    return load\\n\\n\\ndef concat_parts(cache, parts):\\n    return (np.concatenate([cache[f\\\'L{L}_{p}\\\'][0] for L, p in parts], 1), np.concatenate([cache[f\\\'L{L}_{p}\\\'][1] for L, p in parts], 1))\\n\\n\\ndef plain_and_twin(key, Etr, Ete, C):\\n    """Plain head (test-side standardisation) for the blend; centred twin evidence on grouped rows."""\\n    if W[key] > 0:\\n        sc = StandardScaler().fit(Etr)\\n        clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(Etr), y)\\n        Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-6) if len(Ete) >= 2 else sc.transform(Ete)\\n        add_member(key, full_prob(clf.predict_proba(Xte), clf.classes_))\\n    else:\\n        print(f\\\'  {key}: plain head has weight 0 in the 0.563 blend, not fitted\\\', flush=True)\\n    if key in TWIN_W:\\n        csc = StandardScaler().fit(group_centre_dense(Etr, PROMPT))\\n        twin = LogisticRegression(C=C, max_iter=5000).fit(csc.transform(group_centre_dense(Etr, PROMPT)), y)\\n        if GROUP_ROWS.any():\\n            pc = twin.predict_proba(csc.transform(group_centre_dense(Ete[GROUP_ROWS], G_TE[GROUP_ROWS])))\\n            add_member(key + \\\'_twin\\\', full_prob(pc, twin.classes_), rows=GROUP_ROWS, weight=TWIN_W[key])\\n        if TINY:\\n            st_idx, st_g = self_test_rows()\\n            ps = twin.predict_proba(csc.transform(group_centre_dense(Etr[st_idx], st_g)))\\n            print(f\\\'  {key} twin self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(twin.classes_[ps.argmax(1)] == y[st_idx]):.2f}\\\', flush=True)\\n\\n\\ndef gaussian_group_head(name, spec, Xtr, Xte):\\n    """Our addition: fit the shared-prompt Gaussian on the training prompts, score the grouped test rows jointly within their group."""\\n    prm = fit(Xtr, y, PROMPT, dict(GAUSS_SETTING, d=spec[\\\'d\\\']))\\n    prm[\\\'T1\\\'], prm[\\\'Tg\\\'] = spec[\\\'T1\\\'], spec[\\\'Tg\\\']\\n    print(f\\\'  {name}: d {prm["info"]["d"]}, shrinkage e/p {prm["info"]["a_e"]}/{prm["info"]["a_p"]}, lam pct {prm["info"]["lam_pct"]}\\\', flush=True)\\n    if GROUP_ROWS.any():\\n        LP = scores(Xte[GROUP_ROWS], np.unique(G_TE[GROUP_ROWS], return_inverse=True)[1], prm)\\n        add_member(name, np.exp(LP), rows=GROUP_ROWS, weight=spec[\\\'w\\\'])\\n    if TINY:\\n        st_idx, st_g = self_test_rows()\\n        LP = scores(Xtr[st_idx], st_g, prm)\\n        print(f\\\'  {name} self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(LP.argmax(1) == y[st_idx]):.2f}\\\', flush=True)\\n\\n\\nfrom cache_runtime import install_frozen as _cache_install_frozen\\n_cache_install_frozen(globals())\\n\\nfor key, model_type, hidden, parts, extra_parts, C in LM_HEADS:\\n    print(f\\\'[{key}] {model_type} hidden {hidden} parts {parts + extra_parts}\\\', flush=True)\\n    cache = extract_parts(key, parts + extra_parts, decoder_loader(model_type, hidden, max(L for L, _ in parts + extra_parts) + 1))\\n    if G_TE is None:   # the GLM pass comes first: its layer-24 pool defines the test groups\\n        G_TE = mixed_groups(TE, cache[f\\\'L{GROUP_LAYER}_mean\\\'][1])\\n        GROUP_ROWS = eligible(G_TE)\\n        describe_groups(\\\'test groups (content + GLM L24)\\\', G_TE)\\n        if SMOKE:\\n            same_g, same_p = G_TE[:, None] == G_TE[None, :], test[\\\'prompt_id\\\'].to_numpy(dtype=object)[:, None] == test[\\\'prompt_id\\\'].to_numpy(dtype=object)[None, :]\\n            off = ~np.eye(len(TE), dtype=bool)\\n            print(f\\\'  smoke grouping: pair precision {(same_g & same_p & off).sum() / max((same_g & off).sum(), 1):.2f} \\\'\\n                  f\\\'recall {(same_g & same_p & off).sum() / max((same_p & off).sum(), 1):.2f}\\\', flush=True)\\n    CACHED_PARTS[key] = cache\\n    plain_and_twin(key, *concat_parts(cache, parts), C)\\n    for name, spec in GAUSS_HEADS.items():\\n        if spec[\\\'src\\\'] == key:\\n            gaussian_group_head(name, spec, *cache[spec[\\\'part\\\']])\\n    del cache\\n    CACHED_PARTS.pop(key)\\n    gc.collect()\\n\\nfor key, dir_name, parts, C in ENC_HEADS:\\n    print(f\\\'[{key}] {dir_name} parts {parts}\\\', flush=True)\\n    cache = extract_parts(key, parts, encoder_loader(dir_name))\\n    plain_and_twin(key, *concat_parts(cache, parts), C)\\n    for name, spec in GAUSS_HEADS.items():\\n        if spec[\\\'src\\\'] == key:\\n            gaussian_group_head(name, spec, *cache[spec[\\\'part\\\']])\\n    del cache\\n    gc.collect()\\n\\n# ============================================================ 5. LoRA classifiers (inference only)\\n@torch.no_grad()\\ndef score_lora_checkpoints(trunk_spec, pool_layers, checkpoints):\\n    """One truncated trunk, adapters swapped in turn. Plain readout -> blend member; centred readout -> twin on grouped rows;\\n    within-Claude readout -> Claude gate."""\\n    import importlib.util as iu\\n    _find = iu.find_spec   # Kaggle\\\'s torchao build is rejected by peft; hide it (unused here)\\n    iu.find_spec = lambda name, *a, **k: None if name.split(\\\'.\\\')[0] == \\\'torchao\\\' else _find(name, *a, **k)\\n    from peft import PeftModel\\n    import peft.import_utils as pi\\n    pi.is_torchao_available = lambda *a, **k: False\\n    from safetensors.torch import load_file\\n    model_type, H, keep = trunk_spec\\n    base = find_checkpoint(model_type, H)\\n    assert base, f\\\'no {model_type} base checkpoint under /kaggle/input\\\'\\n    cfg = AutoConfig.from_pretrained(base)\\n    cfg.num_hidden_layers, cfg.use_cache = keep, False\\n    if getattr(cfg, \\\'layer_types\\\', None) is not None:\\n        cfg.layer_types = cfg.layer_types[:keep]\\n    trunk = AutoModel.from_pretrained(base, config=cfg, dtype=torch.float16, device_map=\\\'auto\\\').eval()\\n    for key, marker, norms_file, anchor_file in checkpoints:\\n        hits = sorted(glob.glob(\\\'/kaggle/input/\\\' + marker, recursive=True))\\n        assert hits, f\\\'checkpoint for {key} not attached ({marker})\\\'\\n        ck = pathlib.Path(os.path.dirname(hits[0]))\\n        print(f\\\'[{key}] {ck}\\\', flush=True)\\n        tok = AutoTokenizer.from_pretrained(ck / \\\'tokenizer\\\')\\n        tok.padding_side = \\\'right\\\'\\n        if tok.pad_token is None:\\n            tok.pad_token = tok.eos_token\\n        lm = PeftModel.from_pretrained(trunk, ck / \\\'adapter\\\').eval()\\n        state = load_file(str(ck / norms_file))\\n        norms = torch.nn.ModuleList([torch.nn.LayerNorm(H) for _ in pool_layers])\\n        norms.load_state_dict({k[len(\\\'norms.\\\'):]: v for k, v in state.items() if k.startswith(\\\'norms.\\\')})\\n        ro = np.load(ck / \\\'readouts.npz\\\')\\n        assert list(ro[\\\'classes\\\']) == LABELS and list(ro[\\\'pool_layers\\\']) == pool_layers\\n        dev = lm.get_input_embeddings().weight.device\\n        norms = norms.to(dev).float()\\n\\n        def token_ids(texts):\\n            return [ids + [tok.eos_token_id] for ids in tok(list(texts), add_special_tokens=False, truncation=True, max_length=LORA_MAX_LEN)[\\\'input_ids\\\']]\\n\\n        def features(texts, bs):\\n            ids = token_ids(texts)\\n            Z = np.zeros((len(ids), H * len(pool_layers)), dtype=np.float32)\\n            order = np.argsort([len(t) for t in ids], kind=\\\'stable\\\')\\n            for b0 in range(0, len(order), bs):\\n                idx = order[b0:b0 + bs]\\n                batch = tok.pad({\\\'input_ids\\\': [ids[i] for i in idx]}, return_tensors=\\\'pt\\\').to(dev)\\n                with torch.autocast(\\\'cuda\\\', dtype=torch.float16):\\n                    hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states\\n                m = batch[\\\'attention_mask\\\'].unsqueeze(-1).float()\\n                m[:, 0] = 0   # the first token is masked out of the mean, as in training\\n                Z[idx] = torch.cat([nm((hs[L].float() * m.to(hs[L].device)).sum(1).to(dev) / m.sum(1).to(dev))\\n                                    for L, nm in zip(pool_layers, norms)], -1).cpu().numpy()\\n                del hs, batch\\n            return Z\\n\\n        if (ck / anchor_file).exists():   # the checkpoint\\\'s own features for training rows 0..2: our forward pass must reproduce them\\n            saved = np.load(ck / anchor_file)[\\\'features\\\'].astype(np.float32)\\n            ours = features(TR[:3], 3)\\n            rel = float(np.abs(ours - saved).mean() / (np.abs(saved).mean() + 1e-9))\\n            print(f\\\'  parity with the checkpoint anchors: max abs {float(np.abs(ours - saved).max()):.4f}, mean rel {rel:.5f}\\\', flush=True)\\n            assert rel < 0.02, f\\\'{key}: our features do not match the checkpoint\\\'\\n        Z = features(TE, LORA_BS)\\n        add_member(key, softmax_rows(Z @ ro[\\\'plain_coef\\\'].T + ro[\\\'plain_intercept\\\']))\\n        if \\\'claude_coef\\\' in ro.files:\\n            lc = Z @ ro[\\\'claude_coef\\\'].T + ro[\\\'claude_intercept\\\']\\n            GATES[\\\'claude_lora\\\'] = lc - np.log(np.exp(lc - lc.max(1, keepdims=True)).sum(1, keepdims=True)) - lc.max(1, keepdims=True)\\n        if key in LORA_TWIN_W and GROUP_ROWS.any():\\n            Zc = group_centre_dense(Z[GROUP_ROWS], G_TE[GROUP_ROWS])\\n            add_member(key + \\\'_twin\\\', softmax_rows(Zc @ ro[\\\'center_coef\\\'].T + ro[\\\'center_intercept\\\']), rows=GROUP_ROWS, weight=LORA_TWIN_W[key])\\n        trunk = lm.unload()\\n        del lm, Z\\n        gc.collect()\\n        torch.cuda.empty_cache()\\n    del trunk\\n    gc.collect()\\n    torch.cuda.empty_cache()\\n\\n\\nif not SMOKE:\\n    for trunk_spec, pool_layers, checkpoints in LORA_RUNS:\\n        score_lora_checkpoints(trunk_spec, pool_layers, checkpoints)\\n\\n# ============================================================ 6. token-attention heads + UltraBERT\\nclass TokenAttention(torch.nn.Module if torch is not None else object):   # the module of the public training kernels (lucasboesen/name-that-llm-train-tokattn, -tokattn2)\\n    def __init__(self, k, h=4096):\\n        super().__init__()\\n        self.drop, self.proj, self.score, self.out = torch.nn.Dropout(0.2), torch.nn.Linear(h, 64), torch.nn.Linear(64, 1), torch.nn.Linear(128, k)\\n\\n    def forward(self, x, m):\\n        h = torch.tanh(self.proj(self.drop(x)))\\n        a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -1e4), -1).unsqueeze(-1)\\n        mf = m.unsqueeze(-1).float()\\n        return self.out(self.drop(torch.cat([(h * mf).sum(1) / mf.sum(1).clamp(min=1), (a * h).sum(1)], -1)))\\n\\n\\n@torch.no_grad()\\ndef token_attention_probs(texts, head_file, backbone, bs=8):\\n    hits = sorted(glob.glob(\\\'/kaggle/input/**/\\\' + head_file, recursive=True))\\n    assert hits, f\\\'training-kernel output holding {head_file} not attached\\\'\\n    d = torch.load(hits[0], map_location=\\\'cpu\\\')\\n    assert list(d[\\\'classes\\\']) == LABELS\\n    decoder = d.get(\\\'decoder\\\', True)\\n    path = find_checkpoint(\\\'qwen3_5\\\') if backbone == \\\'qwen3_5\\\' else find_checkpoint(None, dir_name=\\\'deberta-v3-large\\\')\\n    assert path, f\\\'backbone {backbone} not attached\\\'\\n    cfg = AutoConfig.from_pretrained(path)\\n    if decoder:\\n        tc = getattr(cfg, \\\'text_config\\\', None) or cfg\\n        tc.num_hidden_layers = d[\\\'keep\\\']\\n        if getattr(tc, \\\'layer_types\\\', None):\\n            tc.layer_types = tc.layer_types[:d[\\\'keep\\\']]\\n        lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map=\\\'auto\\\').eval()\\n    else:\\n        lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()\\n    tok = AutoTokenizer.from_pretrained(path)\\n    tok.padding_side = \\\'right\\\'\\n    if tok.pad_token is None:\\n        tok.pad_token = tok.eos_token\\n    H = d.get(\\\'hidden\\\', 4096)\\n    net = TokenAttention(K, H)\\n    net.load_state_dict(d[\\\'state_dict\\\'])\\n    dev = lm.get_input_embeddings().weight.device\\n    net, mu, sd = net.to(dev).eval(), d[\\\'mu\\\'].float().to(dev), d[\\\'sd\\\'].float().to(dev)\\n    ids = tok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d[\\\'max_len\\\'])[\\\'input_ids\\\']\\n    P = np.zeros((len(texts), K))\\n    order = np.argsort([len(t) for t in ids])\\n    for b0 in range(0, len(order), bs):\\n        idx = order[b0:b0 + bs]\\n        batch = tok.pad({\\\'input_ids\\\': [ids[i] for i in idx]}, return_tensors=\\\'pt\\\').to(dev)\\n        out = lm(input_ids=batch[\\\'input_ids\\\'], attention_mask=batch[\\\'attention_mask\\\'], output_hidden_states=True)\\n        hs = (out.hidden_states if getattr(out, \\\'hidden_states\\\', None) is not None else out.language_model_outputs.hidden_states)[d[\\\'layer\\\']]\\n        n = batch[\\\'attention_mask\\\'].sum(1)\\n        T = max(int(n.max()) - 1, 1)\\n        x = torch.zeros(len(idx), T, H, device=dev)\\n        m = torch.zeros(len(idx), T, dtype=torch.bool, device=dev)\\n        for r in range(len(idx)):   # first token dropped, states rounded through fp16, as in training\\n            k = int(n[r]) - 1\\n            if k > 0:\\n                x[r, :k] = ((torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float() - mu) / sd).clamp(-8, 8)\\n                m[r, :k] = True\\n        P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()\\n        del hs, out, batch\\n    del lm\\n    gc.collect()\\n    torch.cuda.empty_cache()\\n    return P\\n\\n\\n@torch.no_grad()\\ndef ultrabert_probs(texts):\\n    from transformers import AutoModelForSequenceClassification\\n    folds = None\\n    for root in glob.glob(\\\'/kaggle/input/**/ultrabert-380m-ntl\\\', recursive=True):\\n        fs = [f for f in sorted(glob.glob(os.path.join(root, \\\'fold*\\\'))) if os.path.exists(os.path.join(f, \\\'config.json\\\'))\\n              and json.load(open(os.path.join(f, \\\'config.json\\\'))).get(\\\'model_type\\\') == \\\'ultrabert\\\']\\n        if len(fs) == 5:\\n            folds = fs\\n            break\\n    assert folds, \\\'ultrabert-380m-ntl (5 folds) not attached\\\'\\n    tok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=True)\\n    ids = tok(texts, truncation=True, max_length=512)[\\\'input_ids\\\']\\n    order = np.argsort([len(e) for e in ids])\\n    P = np.zeros((len(texts), K))\\n    for fp in folds:\\n        em = AutoModelForSequenceClassification.from_pretrained(fp, trust_remote_code=True, dtype=torch.float16, attn_implementation=\\\'sdpa\\\').to(DEV).eval()\\n        cols = [LAB2I[em.config.id2label[i]] for i in range(len(em.config.id2label))]\\n        for b0 in range(0, len(order), 32):\\n            idx = order[b0:b0 + 32]\\n            b = tok.pad({\\\'input_ids\\\': [ids[j] for j in idx]}, return_tensors=\\\'pt\\\').to(DEV)\\n            P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)\\n        del em\\n        gc.collect()\\n        torch.cuda.empty_cache()\\n    return P\\n\\n\\nif not SMOKE:\\n    for key, head_file, backbone in TOKATTN_HEADS:\\n        if TINY:   # in-sample check on training rows the head was fitted on: must be high, or the checkpoint is not being reproduced\\n            acc = float(np.mean(token_attention_probs(TR[:40], head_file, backbone).argmax(1) == y[:40]))\\n            print(f\\\'  {key} in-sample check on 40 training rows: acc {acc:.2f}\\\', flush=True)\\n            assert acc > 0.5, f\\\'{key} does not reproduce its training fit\\\'\\n        add_member(key, token_attention_probs(TE, head_file, backbone))\\n    add_member(\\\'ultrabert\\\', ultrabert_probs(TE))\\n\\n\'\n# Complementary feature ensemble: source Gaussian stack, no hard family gates.\n_new_labels = list(LABELS)\n_new_prob = softmax(LOGIT.copy())\n_old_namespace = {\'__name__\': \'__gaussian_stack__\'}\nexec(compile(_GAUSSIAN_SOURCE, \'public_gaussian_stack\', \'exec\'), _old_namespace)\nassert list(_old_namespace[\'test\'][\'id\'].astype(str)) == list(test[\'id\'].astype(str))\n# Reuse the complementary stack\'s existing prompt partition, with original test row order.\n_consensus_glm_groups = np.asarray(_old_namespace[\'G_TE\'], dtype=np.int64).copy()\n_consensus_glm_ids = _old_namespace[\'test\'][\'id\'].astype(str).tolist()\nif _consensus_glm_groups.shape != (len(test),):\n    raise ValueError(\'Complementary grouping has an unexpected test-row shape\')\n_old_cols = [_old_namespace[\'LABELS\'].index(label) for label in _new_labels]\n_old_raw = _old_namespace[\'LOGIT\'][:, _old_cols]\n_old_prob = softmax(_old_raw)\n_mix_prob = 0.85 * _new_prob + 0.15 * _old_prob\nassert np.isfinite(_mix_prob).all() and np.allclose(_mix_prob.sum(1), 1.0)\nLOGIT = np.log(np.clip(_mix_prob, 1e-12, 1.0))\nprint(\'BLEND two-stage .85 + original four-Gaussian stack .15; labels aligned; single final Sinkhorn\', flush=True)\ndel _old_namespace, _old_raw, _old_prob, _new_prob, _mix_prob\nimport gc as _gc\n_gc.collect()\ntorch.cuda.empty_cache()\n\n\nnp.savez_compressed(\'/kaggle/working/baseline-pre-atlas.npz\',\n    probabilities=softmax(LOGIT), labels=np.asarray(LABELS), ids=test.id.astype(str).to_numpy(dtype=str),\n    groups=np.asarray(groups,dtype=np.int64), glm_groups=np.asarray(_consensus_glm_groups,dtype=np.int64))\n', 'checkpoint_resolver.py': '"""Reject config-only adapter folders when selecting a real base checkpoint."""\nimport json as _weights_json\nimport os as _weights_os\n\n\ndef _ntl_has_model_weights(directory):\n    for name in (\'model.safetensors\', \'pytorch_model.bin\'):\n        if _weights_os.path.isfile(_weights_os.path.join(directory, name)):\n            return True\n    for name in (\'model.safetensors.index.json\', \'pytorch_model.bin.index.json\'):\n        path = _weights_os.path.join(directory, name)\n        if not _weights_os.path.isfile(path):\n            continue\n        try:\n            with open(path, encoding=\'utf-8\') as stream:\n                mapping = _weights_json.load(stream).get(\'weight_map\', {})\n            shards = set(mapping.values())\n            if shards and all(isinstance(shard, str) and\n                              _weights_os.path.isfile(_weights_os.path.join(directory, shard))\n                              for shard in shards):\n                return True\n        except (OSError, ValueError, TypeError, AttributeError):\n            continue\n    return False\n', 'gauss.py': '"""Shared-prompt Gaussian group head (ported, Apache-2.0, from public Kaggle notebooks by prvsiyan / kirilltym / sangam21).\nTwo-way model x = class mean + prompt effect + noise; exact (n<=3) or cavity-EM (n>3) injective group marginals."""\nimport itertools\nimport numpy as np\nK = 18\nDEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,\n           "iters": 5, "repel": True, "sp_scale": 1.0}\n\n# =================================================================== primitives ====\ndef _lse(A, axis=None, keepdims=False):\n    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)\n    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m\n    if keepdims: return out\n    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())\n\ndef log_softmax(L): return L - _lse(L, axis=1, keepdims=True)\n\ndef _lw_shrinkage(X):\n    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""\n    from sklearn.covariance import ledoit_wolf_shrinkage\n    return float(ledoit_wolf_shrinkage(X, assume_centered=True))\n\ndef _shrunk(S, a):\n    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)\n\ndef _two_way(Z, y, pinv, G, iters=4):\n    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""\n    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)\n    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))\n    for _ in range(iters):\n        R = Z - P[pinv]\n        for c in cls: mu[c] = R[y == c].mean(0)\n        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]\n    return mu, P, cnt_p\n\ndef fit(X, y, prompts, cfg=None):\n    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.\n    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller\'s cross-fit)."""\n    c = dict(DEFAULT); c.update(cfg or {})\n    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape\n    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)\n    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx\n    if c["red"] == "none":\n        V = np.eye(D)\n    else:\n        if c["red"] == "wpca":\n            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]\n        elif c["red"] == "pca": R = Xs\n        else: raise ValueError(c["red"])\n        d = int(min(c["d"], N - G - 1, D))\n        # economy SVD through the (N x N) Gram matrix (N << D for the real views)\n        if N < D:\n            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]\n            V = (R.T @ U) / np.sqrt(w)\n        else:\n            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T\n    Z = Xs @ V; d = Z.shape[1]\n    mu, P, ng = _two_way(Z, y, pinv, G)\n    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)\n    Se_raw = E.T @ E / dof\n    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)\n    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw\n    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])\n    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]\n    # joint diagonalisation: A Se A\' = I, A Sp A\' = diag(lam)\n    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)\n    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li\n    W = V @ A.T\n    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,\n            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}\n\ndef transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]\n\n_MASK = {}\ndef _inj_mask(n, C):\n    if (n, C) not in _MASK:\n        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)\n        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]\n        _MASK[(n, C)] = bad\n    return _MASK[(n, C)]\n\ndef singleton_ll(U, m, lam):\n    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""\n    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)\n\ndef joint_terms(U, m, lam):\n    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""\n    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]\n    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)\n    return a, B\n\ndef joint_table(a, B):\n    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""\n    n, C = a.shape; Q = np.zeros([C] * n)\n    for i in range(n):\n        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)\n    for i, j in itertools.combinations(range(n), 2):\n        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)\n    return Q\n\ndef exact_marginals(Q, T=1.0, inj=True):\n    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""\n    n = Q.ndim; C = Q.shape[0]; Qt = Q / T\n    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)\n    lz = _lse(Qt); out = np.empty((n, C))\n    for i in range(n):\n        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz\n    return out\n\ndef em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):\n    """cavity EM for one group (any n >= 2): returns (n, C) log q."""\n    n = len(U); v = lam / (1 + (n - 1) * lam)\n    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))\n    q = np.exp(log_softmax(L))\n    for _ in range(iters):\n        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)\n        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T\n        if repel:   # mean-field at-most-once: discourage classes the other members already hold\n            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)\n        q = np.exp(log_softmax(L))\n    return log_softmax(L)\n\nLAM_DIFF = 1e6\ndef lam_g(prm, cfg=None):\n    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg \'diff\'): then the joint likelihood\n    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs\n    centred prototypes) and the prompt term cancels exactly."""\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})\n    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]\n\ndef group_logpost(U, prm, T1=None, Tg=None, cfg=None):\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)\n    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg\n    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)\n    if n <= c["n_exact"]:\n        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])\n    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])\n\ndef scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):\n    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""\n    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))\n    sizes = np.bincount(lab); s1 = sizes[lab] == 1\n    T1 = prm["T1"] if T1 is None else T1\n    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)\n    for g in np.where(sizes >= 2)[0]:\n        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)\n    return out\n', 'cache_runtime.py': '"""Checkpoint-bound train features and fitted Gaussian parameters; never cache test predictions."""\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\n\nFORMAT = 1\n_STORE = None\n_IDENTITIES = {}\n\n\ndef digest_json(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\',\', \':\'), ensure_ascii=False).encode()).hexdigest()\n\n\ndef file_sha(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef array_sha(value):\n    value = np.ascontiguousarray(value)\n    if value.dtype.hasobject:\n        return digest_json(value.tolist())\n    result = hashlib.sha256()\n    result.update(str(value.dtype).encode()); result.update(str(value.shape).encode())\n    result.update(memoryview(value).cast(\'B\'))\n    return result.hexdigest()\n\n\ndef train_signature(frame):\n    columns = [\'id\', \'prompt_id\', \'model\', \'text\']\n    if not all(column in frame for column in columns):\n        raise ValueError(\'Official train columns are incomplete\')\n    return digest_json([[str(v) for v in row] for row in frame[columns].itertuples(index=False, name=None)])\n\n\ndef runtime_versions():\n    result = {}\n    for name in [\'numpy\', \'scipy\', \'scikit-learn\', \'torch\', \'transformers\']:\n        try: result[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError: pass\n    return result\n\n\ndef checkpoint_identity(base, adapter=None):\n    """Bind inputs to metadata/tokenizers/heads/adapters and base shard names/sizes.\n\n    Kaggle input references are recorded separately. Full multi-GB base shards are\n    not rehashed on every inference; adapter/head tensors are SHA256 checked.\n    """\n    base = Path(base).resolve()\n    adapter = Path(adapter).resolve() if adapter is not None else None\n    key = (str(base), str(adapter))\n    if key in _IDENTITIES:\n        return _IDENTITIES[key]\n    if not (base / \'config.json\').is_file():\n        raise FileNotFoundError(\'Base config missing\')\n    small = {}\n    names = [\'config.json\', \'model.safetensors.index.json\', \'pytorch_model.bin.index.json\',\n             \'tokenizer.json\', \'tokenizer_config.json\', \'special_tokens_map.json\',\n             \'tokenizer.model\', \'spm.model\', \'vocab.json\', \'merges.txt\', \'added_tokens.json\']\n    for name in names:\n        if (base / name).is_file():\n            small[name] = file_sha(base / name)\n    weights = {}\n    for name in [\'model.safetensors.index.json\', \'pytorch_model.bin.index.json\']:\n        path = base / name\n        if path.is_file():\n            shards = sorted(set(json.loads(path.read_text())[\'weight_map\'].values()))\n            for shard in shards:\n                path = base / shard\n                if not path.is_file():\n                    raise FileNotFoundError(\'A base weight shard is missing\')\n                weights[shard] = path.stat().st_size\n    if not weights:\n        for name in [\'model.safetensors\', \'pytorch_model.bin\']:\n            if (base / name).is_file():\n                weights[name] = (base / name).stat().st_size\n    if not weights:\n        raise FileNotFoundError(\'Refusing a config-only model folder\')\n    ad = {}\n    if adapter is not None:\n        for name in [\'meta.json\', \'heads.pt\', \'heads.safetensors\', \'adapter_config.json\',\n                     \'adapter_model.safetensors\', \'adapter/adapter_config.json\',\n                     \'adapter/adapter_model.safetensors\']:\n            if (adapter / name).is_file():\n                ad[name] = file_sha(adapter / name)\n        if not any(name.endswith(\'adapter_model.safetensors\') for name in ad):\n            raise FileNotFoundError(\'An actual LoRA adapter is required\')\n        if not any(name in ad for name in [\'heads.pt\', \'heads.safetensors\']):\n            raise FileNotFoundError(\'An actual pretrained classifier head is required\')\n    result = {\'base_metadata_sha256\': small, \'base_weight_bytes\': weights, \'adapter_sha256\': ad}\n    _IDENTITIES[key] = result\n    return result\n\n\ndef _jsonable(value):\n    if isinstance(value, dict): return {k: _jsonable(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)): return [_jsonable(v) for v in value]\n    if isinstance(value, np.generic): return value.item()\n    return value\n\n\nclass TrainCache:\n    def __init__(self, root, contract, train, write=False):\n        self.root = Path(root)\n        self.manifest_path = self.root / \'train-cache-manifest.json\'\n        self.write = write\n        self.contract = contract\n        self.train_sha = train_signature(train)\n        self.events = []\n        if write: self.root.mkdir(parents=True, exist_ok=True)\n        if self.manifest_path.is_file():\n            self.manifest = json.loads(self.manifest_path.read_text(encoding=\'utf-8\'))\n            if (self.manifest[\'format\'] != FORMAT or self.manifest[\'contract\'] != contract or\n                    self.manifest[\'train_sha256\'] != self.train_sha):\n                raise ValueError(\'Train cache source contract or official training data differs\')\n            if self.manifest[\'runtime_versions\'] != runtime_versions():\n                raise ValueError(\'Cache preparation/inference software versions differ\')\n            if not write and self.manifest[\'status\'] != \'COMPLETE\':\n                raise ValueError(\'Incomplete train cache is not usable for inference\')\n        elif write:\n            self.manifest = {\'format\': FORMAT, \'contract\': contract, \'train_sha256\': self.train_sha,\n                             \'train_rows\': len(train), \'status\': \'BUILDING\', \'features\': {}, \'fits\': {},\n                             \'runtime_versions\': runtime_versions(),\n                             \'parity\': {}, \'created_at_utc\': time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime())}\n            self.flush()\n        else:\n            raise FileNotFoundError(\'Verified train-cache-manifest.json is required\')\n\n    def flush(self):\n        if not self.write: return\n        target = self.manifest_path.with_suffix(\'.tmp\')\n        target.write_text(json.dumps(_jsonable(self.manifest), indent=2), encoding=\'utf-8\')\n        target.replace(self.manifest_path)\n\n    def save_features(self, key, arrays, recipe, identity, runtime=None):\n        if not self.write: raise RuntimeError(\'Read-only cache\')\n        for value in arrays.values():\n            if value.ndim != 2 or len(value) != self.manifest[\'train_rows\'] or not np.isfinite(value).all():\n                raise ValueError(\'Invalid train feature matrix\')\n        filename = \'features-\' + key + \'.npz\'\n        path = self.root / filename\n        np.savez_compressed(path, **arrays)\n        self.manifest[\'features\'][key] = {\'file\': filename, \'sha256\': file_sha(path),\n            \'recipe\': _jsonable(recipe), \'checkpoint\': identity,\n            \'runtime\': runtime or {},\n            \'arrays\': {k: {\'shape\': list(v.shape), \'dtype\': str(v.dtype), \'sha256\': array_sha(v)} for k, v in arrays.items()}}\n        self.flush()\n        loaded = self.load_features(key, recipe, identity)\n        if not all(np.array_equal(loaded[k], value) for k, value in arrays.items()):\n            raise ValueError(\'Train feature serialization changed values\')\n        self.manifest[\'parity\'][\'features-\' + key] = {\'passed\': True, \'method\': \'exact-array-round-trip\'}\n        self.flush()\n\n    def load_features(self, key, recipe, identity):\n        record = self.manifest[\'features\'][key]\n        if record[\'recipe\'] != _jsonable(recipe) or record[\'checkpoint\'] != identity:\n            raise ValueError(\'Cached feature recipe/checkpoint differs: \' + key)\n        path = self.root / record[\'file\']\n        if file_sha(path) != record[\'sha256\']: raise ValueError(\'Changed feature artifact: \' + key)\n        with np.load(path, allow_pickle=False) as pack:\n            arrays = {name: pack[name] for name in pack.files}\n        if set(arrays) != set(record[\'arrays\']): raise ValueError(\'Feature parts differ\')\n        for name, value in arrays.items():\n            meta = record[\'arrays\'][name]\n            if (list(value.shape) != meta[\'shape\'] or str(value.dtype) != meta[\'dtype\'] or\n                    array_sha(value) != meta[\'sha256\'] or not np.isfinite(value).all()):\n                raise ValueError(\'Invalid cached train features\')\n        self.events.append({\'kind\': \'train_features\', \'key\': key})\n        return arrays\n\n    def fit(self, namespace, original_fit, X, y, prompts, cfg=None):\n        identity = {\'namespace\': namespace, \'features\': array_sha(X), \'y\': array_sha(y),\n                    \'prompts\': digest_json([str(v) for v in prompts]), \'cfg\': _jsonable(cfg)}\n        key = digest_json(identity)\n        if self.write:\n            result = original_fit(X, y, prompts, cfg)\n            arrays = {k: v for k, v in result.items() if isinstance(v, np.ndarray)}\n            metadata = {k: _jsonable(v) for k, v in result.items() if k not in arrays}\n            filename = \'gaussian-\' + key + \'.npz\'\n            np.savez_compressed(self.root / filename, **arrays)\n            record = {\'file\': filename, \'sha256\': file_sha(self.root / filename), \'identity\': identity,\n                      \'metadata\': metadata, \'array_names\': sorted(arrays)}\n            self.manifest[\'fits\'][key] = record\n            self.flush()\n            loaded = self._load_fit(key)\n            if metadata != {k: _jsonable(v) for k, v in loaded.items() if k not in arrays}:\n                raise ValueError(\'Fitted metadata differs after serialization\')\n            if not all(np.array_equal(result[k], loaded[k]) for k in arrays):\n                raise ValueError(\'Fitted tensors differ after serialization\')\n            self.manifest[\'parity\'][\'fit-\' + key] = {\'passed\': True, \'method\': \'exact-parameter-round-trip\'}\n            self.flush()\n            return result\n        if key not in self.manifest[\'fits\']:\n            raise ValueError(\'Fitted Gaussian cache does not match inputs: \' + namespace)\n        self.events.append({\'kind\': \'fitted_gaussian\', \'namespace\': namespace, \'key\': key})\n        return self._load_fit(key)\n\n    def _load_fit(self, key):\n        record = self.manifest[\'fits\'][key]\n        path = self.root / record[\'file\']\n        if file_sha(path) != record[\'sha256\']: raise ValueError(\'Changed fitted Gaussian artifact\')\n        with np.load(path, allow_pickle=False) as pack:\n            result = {name: pack[name] for name in pack.files}\n        if sorted(result) != record[\'array_names\'] or not all(np.isfinite(v).all() for v in result.values()):\n            raise ValueError(\'Invalid fitted Gaussian arrays\')\n        return {**result, **record[\'metadata\']}\n\n\ndef get_store():\n    global _STORE\n    if _STORE is None:\n        import pandas as pd\n        contract = json.loads(Path(\'/kaggle/working/cache-contract.json\').read_text())\n        paths = sorted(Path(\'/kaggle/input\').rglob(\'train-cache-manifest.json\'))\n        if len(paths) != 1: raise ValueError(\'Attach exactly one completed train feature cache\')\n        locations = [Path(\'/kaggle/input/competitions/name-that-llm/train.csv\'),\n                     Path(\'/kaggle/input/name-that-llm/train.csv\')]\n        path = next(p for p in locations if p.is_file())\n        frame = pd.read_csv(path); frame.text = frame.text.fillna(\'\').astype(str)\n        _STORE = TrainCache(paths[0].parent, contract, frame)\n    return _STORE\n\n\ndef main_recipe(name, bs):\n    return {\'kind\': \'nishant-multi-layer-mean\', \'name\': name, \'batch_size\': bs}\n\n\ndef frozen_recipe(key, parts, decoder, bs):\n    return {\'kind\': \'frozen-pools\', \'name\': key, \'parts\': [list(v) for v in parts],\n            \'decoder\': bool(decoder), \'batch_size\': bs, \'max_len\': 512}\n\n\ndef dhruv_recipe(name, spec):\n    return {\'kind\': \'dhruv-multi-layer-mean\', \'name\': name, \'spec\': spec, \'max_len\': 384}\n\n\ndef install_main(namespace):\n    store = get_store()\n    original_features = namespace[\'lora_features\']\n    train_texts = namespace[\'train\'].text.astype(str).tolist()\n    def features(ckpt, base, texts, bs):\n        name = Path(ckpt).name\n        if name == \'lora8b_v1\' and len(texts) >= len(train_texts) and list(texts[-len(train_texts):]) == train_texts:\n            cached = store.load_features(\'main-\' + name, main_recipe(name, bs), checkpoint_identity(base, ckpt))[\'Z\']\n            test_count = len(texts) - len(train_texts)\n            if test_count <= 0: raise ValueError(\'Expected test features followed by official train\')\n            test, heads = original_features(ckpt, base, texts[:test_count], bs)\n            return np.concatenate([test, cached], axis=0), heads\n        return original_features(ckpt, base, texts, bs)\n    namespace[\'lora_features\'] = features\n    original_fit = namespace[\'fit\']\n    namespace[\'fit\'] = lambda X, y, prompts, cfg=None: store.fit(\'main\', original_fit, X, y, prompts, cfg)\n\n\ndef install_frozen(namespace):\n    store = get_store()\n    def extract(key, parts, loader):\n        if namespace[\'SMOKE\']: raise ValueError(\'Synthetic features cannot use a production cache\')\n        model, tok, decoder, bs = loader()\n        identity = checkpoint_identity(tok.name_or_path)\n        train = store.load_features(\'frozen-\' + key, frozen_recipe(key, parts, decoder, bs), identity)\n        # The source frozen extractor can switch permanently to BF16 after a\n        # nonfinite FP16 training batch. Preserve that state for test inference.\n        dtype_name = store.manifest[\'features\'][\'frozen-\' + key][\'runtime\'].get(\'dtype_after_train\')\n        if dtype_name:\n            dtype = getattr(namespace[\'torch\'], dtype_name)\n            if next(model.parameters()).dtype != dtype: model.to(dtype=dtype)\n        test = namespace[\'pooled_states\'](model, tok, namespace[\'TE\'], parts, decoder, bs)\n        del model, tok\n        gc.collect()\n        namespace[\'torch\'].cuda.empty_cache()\n        return {key: (train[key], test[key]) for key in train}\n    namespace[\'extract_parts\'] = extract\n    original_fit = namespace[\'fit\']\n    namespace[\'fit\'] = lambda X, y, prompts, cfg=None: store.fit(\'frozen\', original_fit, X, y, prompts, cfg)\n\n\ndef dhruv_train_features(name, spec, base, adapter):\n    return get_store().load_features(\'dhruv-\' + name, dhruv_recipe(name, spec), checkpoint_identity(base, adapter))[\'Z\']\n\n\ndef strict_pretrained_loading():\n    """Reject absent pretrained weights while permitting deliberately omitted tail layers."""\n    from transformers import AutoModel\n    original = AutoModel.from_pretrained\n    def checked(*args, **kwargs):\n        requested = kwargs.get(\'output_loading_info\', False)\n        kwargs[\'output_loading_info\'] = True\n        model, info = original(*args, **kwargs)\n        if info.get(\'missing_keys\') or info.get(\'mismatched_keys\') or info.get(\'error_msgs\'):\n            raise RuntimeError(\'Refusing incomplete pretrained weights\')\n        return (model, info) if requested else model\n    AutoModel.from_pretrained = staticmethod(checked)\n', 'cache-contract.json': '{\n  "version": 1,\n  "source_hybrid_script_version_id": 361021710,\n  "source_notebook_sha256": "c208c57567cec56cae655256644d1fe7eedeb9c629601692298b0bafafd02aac",\n  "modules_sha256": {\n    "hybrid_math.py": "000c483559decb720709455f1e3950bf8c53155af3679db1eb44b7f93e775cd3",\n    "candidate_runtime.py": "05aa6c4098e360092e94162e78a578d73a19512c69b9d9e45b142f532d826368",\n    "atlas_runtime.py": "d3ac34a5fc5d339f043c194531c464d6251a07a4384a109c358f028920202fbd",\n    "baseline_runtime.py": "7b60bd1c0985de7159d10e52f8722d262f10ab8e706106eaa5e8f526c05cb865",\n    "checkpoint_resolver.py": "5254f7d1bafa97500eb02ea9db33811ce5c22bb299a1af35f6c7d17bc0e11a91",\n    "gauss.py": "908b57113f1a7b8ebdbf204c6be87cbc385f765ba50ac07b4e270c64b7cf1000"\n  },\n  "cache_runtime_sha256": "e13e79e2b6de234c8e6d9daec91c6e7abea5b21d0dad197b5c305892c86025f1",\n  "expected_features": [\n    "main-lora8b_v1",\n    "frozen-glm9b",\n    "frozen-q3_8b",\n    "frozen-q25_7b",\n    "frozen-q3_4b",\n    "frozen-debv3l",\n    "frozen-electl",\n    "frozen-desk",\n    "dhruv-ts_q3b8",\n    "dhruv-ts_q3b14"\n  ],\n  "expected_gaussian_fits": 7,\n  "source_checkpoint_datasets": [\n    "dhruvtibarewal/nlm-l24-adapters/7",\n    "nishantdahal/ntl-two-stage-lora"\n  ]\n}', 'soft_constraints.py': '"""MRF-inspired finite cannot-link penalties on the final ensemble probabilities.\n\nThis is an adaptation of Basu, Bilenko and Mooney (KDD 2004), not a reproduction\nof HMRF-KMEANS. The graph is a set of disjoint weighted cliques formed by the\nintersections of two existing prompt partitions. A uniform pair penalty inside\neach clique permits exact MAP decoding with repeated label slots.\n"""\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\n\n\nSOFT_MRF_DEFAULTS = {\n    \'penalty_strength\': 0.40,\n    \'min_group_size\': 2,\n    \'max_parent_size\': 18,\n    \'min_group_confidence\': 0.50,\n    \'min_energy_improvement\': 0.05,\n    \'min_configuration_gap\': 0.05,\n    \'min_assigned_probability\': 0.01,\n    \'min_relative_probability\': 1.0 / 3.0,\n    \'max_mean_log_loss\': 0.15,\n    \'max_assigned_rank\': 3,\n}\n\n\ndef _soft_energy(unary, labels, penalty):\n    counts = np.bincount(labels, minlength=unary.shape[1])\n    return float(unary[np.arange(len(labels)), labels].sum()\n                 + penalty * (counts * (counts - 1) / 2).sum())\n\n\ndef _solve_soft_clique(unary, penalty):\n    """Exact optimum and next distinct *label configuration*, not slot permutation.\n\nGiving the j-th use of label c a cost penalty*j realizes the total pair penalty\npenalty*count_c*(count_c-1)/2. Every row can use every label, so duplicates remain\npossible. Forbid all slots of a selected label for each row to find the cheapest\ndistinct labeling. At most n+1 small assignment solves for n <= 18.\n"""\n    rows, classes = unary.shape\n    slot_labels = np.repeat(np.arange(classes), rows)\n    extra_cost = penalty * np.tile(np.arange(rows), classes)\n    cost = unary[:, slot_labels] + extra_cost[None, :]\n    row_index, slot_index = linear_sum_assignment(cost)\n    if not np.array_equal(row_index, np.arange(rows)):\n        raise AssertionError(\'Soft assignment failed to cover every group row\')\n    chosen = slot_labels[slot_index]\n    optimum = _soft_energy(unary, chosen, penalty)\n    if not np.isclose(float(cost[row_index, slot_index].sum()), optimum, atol=1e-9, rtol=0):\n        raise AssertionError(\'Repeated label slots disagree with the pairwise objective\')\n    second = np.inf\n    for row, label in enumerate(chosen):\n        alternative = cost.copy()\n        alternative[row, slot_labels == label] = np.inf\n        alt_rows, alt_slots = linear_sum_assignment(alternative)\n        second = min(second, float(alternative[alt_rows, alt_slots].sum()))\n    return chosen, optimum, max(0.0, float(second - optimum))\n\n\ndef soft_constraint_decode(probabilities, tfidf_groups, glm_groups, config=None):\n    """Return guarded MAP labels, retaining probabilities and rejected blocks intact."""\n    settings = dict(SOFT_MRF_DEFAULTS)\n    if config:\n        unknown = set(config) - set(settings)\n        if unknown:\n            raise ValueError(f\'Unknown soft-MRF settings: {sorted(unknown)}\')\n        settings.update(config)\n    for name in [\'penalty_strength\', \'min_energy_improvement\', \'min_configuration_gap\',\n                 \'max_mean_log_loss\']:\n        if not np.isfinite(settings[name]) or settings[name] < 0:\n            raise ValueError(f\'{name} must be finite and nonnegative\')\n    if settings[\'min_configuration_gap\'] <= 0:\n        raise ValueError(\'The configuration gap must be positive to reject tied labelings\')\n    for name in [\'min_group_confidence\', \'min_assigned_probability\', \'min_relative_probability\']:\n        if not 0 < settings[name] <= 1:\n            raise ValueError(f\'{name} must be in (0,1]\')\n    if (not isinstance(settings[\'min_group_size\'], int)\n            or not isinstance(settings[\'max_parent_size\'], int)\n            or not 2 <= settings[\'min_group_size\'] <= settings[\'max_parent_size\'] <= 18):\n        raise ValueError(\'Expected 2 <= min_group_size <= max_parent_size <= 18\')\n    if not isinstance(settings[\'max_assigned_rank\'], int) or settings[\'max_assigned_rank\'] < 1:\n        raise ValueError(\'The rank limit must be a positive integer\')\n    probability = np.asarray(probabilities, dtype=np.float64)\n    if (probability.ndim != 2 or len(probability) == 0 or probability.shape[1] < 2\n            or not np.isfinite(probability).all() or (probability < 0).any()\n            or not np.allclose(probability.sum(1), 1, atol=1e-6, rtol=0)):\n        raise ValueError(\'Expected finite, nonnegative, normalized probabilities\')\n    rows, classes = probability.shape\n    partitions = []\n    sizes = []\n    for name, values in [(\'tfidf\', tfidf_groups), (\'glm\', glm_groups)]:\n        values = np.asarray(values)\n        if values.shape != (rows,) or not np.issubdtype(values.dtype, np.integer):\n            raise ValueError(f\'{name} grouping must contain one integer ID per test row\')\n        _, inverse, counts = np.unique(values, return_inverse=True, return_counts=True)\n        partitions.append(inverse)\n        sizes.append(counts[inverse])\n    intersections, block_index = np.unique(np.column_stack(partitions), axis=0, return_inverse=True)\n    baseline = probability.argmax(1)\n    selected = baseline.copy()\n    decisions = []\n    for block in range(len(intersections)):\n        members = np.flatnonzero(block_index == block)\n        count = len(members)\n        parent_a, parent_b = int(sizes[0][members[0]]), int(sizes[1][members[0]])\n        confidence = count / (parent_a + parent_b - count)\n        penalty = float(settings[\'penalty_strength\'] * confidence)\n        record = {\'block\':int(block), \'rows\':int(count), \'tfidf_parent_rows\':parent_a,\n                  \'glm_parent_rows\':parent_b, \'confidence\':float(confidence),\n                  \'pair_penalty\':penalty, \'accepted\':False, \'changed_rows\':0}\n        if count < settings[\'min_group_size\']:\n            record[\'reason\'] = \'singleton_or_small_intersection\'\n        elif max(parent_a, parent_b) > settings[\'max_parent_size\']:\n            record[\'reason\'] = \'oversized_parent_group\'\n        elif confidence < settings[\'min_group_confidence\']:\n            record[\'reason\'] = \'low_grouping_confidence\'\n        elif penalty == 0:\n            record[\'reason\'] = \'zero_penalty\'\n        else:\n            old = baseline[members]\n            collisions = count - len(np.unique(old))\n            record[\'baseline_duplicate_labels\'] = int(collisions)\n            if not collisions:\n                record[\'reason\'] = \'no_label_collision\'\n                decisions.append(record)\n                continue\n            group_probability = probability[members]\n            unary = -np.log(np.clip(group_probability, 1e-12, 1))\n            candidate, energy, gap = _solve_soft_clique(unary, penalty)\n            baseline_energy = _soft_energy(unary, old, penalty)\n            improvement = baseline_energy - energy\n            if improvement < -1e-8:\n                raise AssertionError(\'Exact soft assignment increased the objective\')\n            assigned = group_probability[np.arange(count), candidate]\n            original = group_probability[np.arange(count), old]\n            relative = assigned / original\n            ranks = (group_probability > assigned[:, None]).sum(1) + 1\n            mean_loss = float(np.log(np.maximum(original, 1e-300)\n                                     / np.maximum(assigned, 1e-300)).mean())\n            changed = int(np.count_nonzero(candidate != old))\n            record.update({\'baseline_energy\':baseline_energy, \'candidate_energy\':energy,\n                           \'energy_improvement\':float(improvement), \'configuration_gap\':float(gap),\n                           \'min_assigned_probability\':float(assigned.min()),\n                           \'min_relative_probability\':float(relative.min()),\n                           \'max_assigned_rank\':int(ranks.max()), \'mean_log_loss\':mean_loss,\n                           \'candidate_changed_rows\':changed,\n                           \'candidate_duplicate_labels\':int(count - len(np.unique(candidate)))})\n            if not changed:\n                record[\'reason\'] = \'baseline_is_soft_optimum\'\n            elif improvement < settings[\'min_energy_improvement\']:\n                record[\'reason\'] = \'insufficient_energy_improvement\'\n            elif gap < settings[\'min_configuration_gap\']:\n                record[\'reason\'] = \'ambiguous_label_configuration\'\n            elif assigned.min() < settings[\'min_assigned_probability\']:\n                record[\'reason\'] = \'assigned_probability_too_low\'\n            elif relative.min() < settings[\'min_relative_probability\']:\n                record[\'reason\'] = \'individual_probability_loss\'\n            elif ranks.max() > settings[\'max_assigned_rank\']:\n                record[\'reason\'] = \'assigned_label_rank\'\n            elif mean_loss > settings[\'max_mean_log_loss\']:\n                record[\'reason\'] = \'group_probability_loss\'\n            else:\n                selected[members] = candidate\n                record.update({\'accepted\':True, \'reason\':\'accepted\', \'changed_rows\':changed})\n        decisions.append(record)\n    reasons = {}\n    for item in decisions:\n        reasons[item[\'reason\']] = reasons.get(item[\'reason\'], 0) + 1\n    report = {\'method\':\'weighted-cannot-link-mrf-exact-clique-map-v1\',\n              \'paper\':\'https://www.cs.utexas.edu/~ml/papers/semi-kdd-04.pdf\',\n              \'settings\':settings, \'rows\':int(rows), \'classes\':int(classes),\n              \'intersection_blocks\':int(len(intersections)),\n              \'accepted_blocks\':sum(int(item[\'accepted\']) for item in decisions),\n              \'changed_rows\':int(np.count_nonzero(selected != baseline)),\n              \'block_reasons\':reasons, \'thresholds_accuracy_validated\':False,\n              \'full_hmrf_kmeans_reproduction\':False, \'block_decisions\':decisions}\n    return selected, report\n', 'atlas_exact_runtime.py': '"""The 0.608 notebook\'s original all-row ATLAS predict call, batch size one."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\n\nimport numpy as np\nimport pandas as pd\nfrom candidate_runtime import find_base\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--request\', required=True)\n    parser.add_argument(\'--output\', required=True)\n    args = parser.parse_args()\n    from cache_runtime import strict_pretrained_loading\n    strict_pretrained_loading()\n    request = json.loads(Path(args.request).read_text())\n    frame = pd.read_csv(request[\'test_path\'])\n    frame.text = frame.text.fillna(\'\').astype(str)\n    roots = list(Path(\'/kaggle/input\').rglob(\'atlas_bundle.json\'))\n    if len(roots) != 1:\n        raise ValueError(\'Exactly one ATLAS bundle is required\')\n    root = roots[0].parent\n    meta = json.loads(roots[0].read_text())\n    if meta[\'files\'][\'adapters.safetensors\'] != \'29e9b32e09679ae4e3b2cdf0729c7011ecee730fdb0f20982593af0b40806106\':\n        raise ValueError(\'ATLAS checkpoint differs from the scored source\')\n    for name, expected in meta[\'files\'].items():\n        with (root / name).open(\'rb\') as stream:\n            if hashlib.file_digest(stream, \'sha256\').hexdigest() != expected:\n                raise ValueError(\'Changed ATLAS asset: \' + name)\n    sys.path.insert(0, str(root))\n    from ntl_atlas.inference import predict\n    from ntl_rlcd.data import LABELS\n    base = find_base(\'/kaggle/input\', dict(model_type=\'qwen3\', hidden_size=4096, num_hidden_layers=36))\n    probability = predict(frame.text.tolist(), base, root, batch_size=1)\n    if (probability.shape != (len(frame), 18) or not np.isfinite(probability).all()\n            or (probability < 0).any() or not np.allclose(probability.sum(1), 1, atol=1e-6)):\n        raise ValueError(\'Invalid full ATLAS output\')\n    np.savez_compressed(args.output, rows=np.arange(len(frame)), probabilities=probability,\n                        labels=np.asarray(LABELS), ids=frame.id.astype(str).to_numpy(dtype=str))\n', 'calibration_runtime.py': '"""Fit one SVC temperature from prompt-grouped out-of-fold decision scores.\n\nOnly classifier heads are fitted. This module never loads or trains an LLM.\nKnown prompt aliases are connected through exact normalized duplicate texts.\nUnknown semantic duplicates can remain; no claim of fully independent ensemble\nvalidation is made. Temperature-fit NLL is not a held-out final-ensemble score.\n"""\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy.optimize import minimize_scalar\nfrom scipy.special import logsumexp\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.svm import LinearSVC\n\n\ndef normalize_text(value):\n    return re.sub(r\'\\s+\', \' \', str(value).strip())\n\n\ndef connected_prompt_groups(raw_frames, pool, train):\n    parent = {}\n\n    def find(value):\n        parent.setdefault(value, value)\n        while parent[value] != value:\n            parent[value] = parent[parent[value]]\n            value = parent[value]\n        return value\n\n    def union(a, b):\n        a, b = find(a), find(b)\n        if a != b:\n            parent[max(a, b)] = min(a, b)\n\n    text_owner = {}\n    frames = [(f\'external-{i}\', frame, \'prompt_key\') for i, frame in enumerate(raw_frames)]\n    frames.append((\'official\', train, \'prompt_id\'))\n    for namespace, frame, column in frames:\n        if column not in frame or frame[column].isna().any():\n            raise ValueError(\'Prompt IDs are required for grouped SVC calibration\')\n        for text, prompt in zip(frame.text, frame[column]):\n            key = namespace + \':\' + str(prompt)\n            find(key)\n            normalized = normalize_text(text)\n            if normalized in text_owner:\n                union(key, text_owner[normalized])\n            else:\n                text_owner[normalized] = key\n    texts = pool.text.tolist() + train.text.tolist()\n    groups = np.asarray([find(text_owner[normalize_text(text)]) for text in texts], dtype=str)\n    if len(np.unique(groups)) < 3:\n        raise ValueError(\'At least three independent known prompt components are required\')\n    return groups\n\n\ndef fit_temperature(scores, targets):\n    scores = np.asarray(scores, dtype=np.float64)\n    targets = np.asarray(targets, dtype=int)\n    if (scores.ndim != 2 or len(scores) != len(targets) or not len(scores)\n            or not np.isfinite(scores).all()\n            or (targets < 0).any() or (targets >= scores.shape[1]).any()):\n        raise ValueError(\'Invalid temperature calibration input\')\n\n    def loss(log_temperature):\n        scaled = scores / np.exp(log_temperature)\n        return float(np.mean(logsumexp(scaled, axis=1) - scaled[np.arange(len(targets)), targets]))\n\n    result = minimize_scalar(loss, bounds=(np.log(0.25), np.log(8.0)), method=\'bounded\',\n                             options={\'maxiter\': 64, \'xatol\': 1e-5})\n    if not result.success or not np.isfinite(result.fun):\n        raise RuntimeError(\'Temperature optimization did not converge\')\n    original = loss(0.0)\n    chosen = float(result.x) if result.fun <= original else 0.0\n    return float(np.exp(chosen)), {\'nll_before\': original, \'nll_after_fit\': loss(chosen),\n                                  \'fit_rows\': len(targets), \'optimizer_iterations\': int(result.nfev)}\n\n\ndef fit_svc_temperature(matrix, targets, pool, train, raw_frames, labels, output_dir=None):\n    targets = np.asarray(targets, dtype=int)\n    groups = connected_prompt_groups(raw_frames, pool, train)\n    if matrix.shape[0] != len(groups) or len(targets) != len(groups):\n        raise ValueError(\'SVC calibration row order differs\')\n    splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)\n    oof = np.full((len(targets), len(labels)), np.nan, dtype=np.float64)\n    folds = np.full(len(targets), -1, dtype=int)\n    reports = []\n    base_weight = np.r_[np.full(len(pool), 0.5), np.ones(len(train))]\n    for fold, (fitting, validation) in enumerate(splitter.split(matrix, targets, groups)):\n        overlap = set(groups[fitting]) & set(groups[validation])\n        if overlap:\n            raise AssertionError(\'Prompt component crossed a calibration fold\')\n        weight = base_weight[fitting].copy()\n        totals = np.bincount(targets[fitting], weights=weight, minlength=len(labels))\n        if (totals <= 0).any():\n            raise ValueError(\'A calibration fold is missing a model class\')\n        weight *= totals.mean() / totals[targets[fitting]]\n        model = LinearSVC(C=0.5, max_iter=3000, random_state=42)\n        model.fit(matrix[fitting], targets[fitting], sample_weight=weight)\n        if model.classes_.tolist() != list(range(len(labels))):\n            raise ValueError(\'Calibration class order differs\')\n        oof[validation] = model.decision_function(matrix[validation])\n        folds[validation] = fold\n        report = {\'fold\': fold, \'fit_rows\': len(fitting), \'validation_rows\': len(validation),\n                  \'known_prompt_overlap\': len(overlap)}\n        reports.append(report)\n        print(\'SVC calibration fold\', fold + 1, \'/3 complete\', report, flush=True)\n        del model\n    if not np.isfinite(oof).all() or (folds < 0).any():\n        raise AssertionError(\'Out-of-fold calibration is incomplete\')\n    official = np.arange(len(pool), len(targets))\n    temperature, loss_report = fit_temperature(oof[official], targets[official])\n    report = {\'method\': \'single-temperature-on-official-rows-from-three-prompt-grouped-SVC-folds\',\n              \'temperature\': temperature, \'folds\': reports, **loss_report,\n              \'known_prompt_components\': int(len(np.unique(groups))),\n              \'known_prompt_overlap\': 0, \'qwen_or_lora_training\': False,\n              \'scope\': \'new-SVC-only; baseline-Qwen-ensemble-is-not-calibrated\',\n              \'nll_after_fit_is_independent_validation\': False,\n              \'unidentified_semantic_prompt_duplicates_may_remain\': True,\n              \'temperature_transferred_from_fold_estimators_to_full_SVC\': True,\n              \'labels\': list(labels),\n              \'group_order_sha256\': hashlib.sha256(json.dumps(groups.tolist()).encode()).hexdigest()}\n    if output_dir is not None:\n        directory = Path(output_dir)\n        (directory / \'svc-calibration-report.json\').write_text(json.dumps(report, indent=2))\n        np.savez_compressed(directory / \'svc-calibration-oof.npz\',\n                            official_scores=oof[official], targets=targets[official],\n                            folds=folds[official], groups=groups[official],\n                            ids=train.id.astype(str).to_numpy(dtype=str), labels=np.asarray(labels))\n    return temperature, report\n'}
for _name,_text in _PREPARED_FILES.items():
    Path('/kaggle/working',_name).write_text(_text,encoding='utf-8')
from cache_runtime import get_store
_cache=get_store()
print('Verified train cache:',len(_cache.manifest['features']),'feature sets,',len(_cache.manifest['fits']),'Gaussian fits',flush=True)


In [ ]:
_NTL_PUBLIC_CACHE = {'ids': ['00353fb1c541', '005f5a9a6139', '0062a53d5487'], 'labels': ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'], 'original_probabilities': [[1.1242558941320582e-05, 1.1257320009241148e-05, 6.560253274581878e-06, 7.582599635513927e-06, 2.7575073460717262e-05, 9.348367690922545e-07, 0.00015695290137938018, 1.3673308825306315e-05, 9.643450423787116e-05, 0.0005274832074752567, 0.6109648776461174, 9.379225795690172e-06, 0.3610424201146699, 0.0067895272367588504, 0.00228973490646004, 0.010248930869063649, 0.0031484256774009286, 0.004647007759725213], [0.05050138709078175, 0.15880513412850147, 0.560435791719202, 0.20342662052324267, 0.019047727706038027, 1.3634830466520401e-05, 1.3921829095334894e-05, 1.2076136713404214e-05, 2.3646527110396513e-05, 1.4047724891860568e-05, 3.402141700399624e-05, 0.0004171340072734819, 1.2575115700707293e-05, 0.0009480558830294252, 0.003076262460726265, 0.00015398361037291373, 0.0013650972429234034, 0.001698882046926509], [3.972346123718417e-07, 9.938242960166919e-06, 5.9289212481127554e-06, 3.3874709820914374e-06, 6.393221835285003e-07, 0.5636713665599524, 0.1987081942702031, 0.1920650909868019, 0.04413799234432296, 0.0005569097354765072, 0.0005972011854483641, 2.930219300590201e-06, 1.0152634630751566e-05, 1.3689820871513115e-05, 3.2389665376430546e-05, 4.9889417848560036e-05, 3.935525152375669e-05, 9.45467162567448e-05]], 'atlas_probabilities': [[0.05339685082435608, 0.050564587116241455, 0.05125558003783226, 0.04949112609028816, 0.051854949444532394, 0.051411911845207214, 0.06321686506271362, 0.054645996540784836, 0.05324713885784149, 0.05988667532801628, 0.05718111991882324, 0.05871433764696121, 0.06423895061016083, 0.05650314316153526, 0.05630448833107948, 0.0598711259663105, 0.057604141533374786, 0.05061103031039238], [0.06367364525794983, 0.0630355253815651, 0.06480997800827026, 0.06549423187971115, 0.06374172866344452, 0.0484013706445694, 0.050788864493370056, 0.046870529651641846, 0.04763931781053543, 0.049332041293382645, 0.0533287487924099, 0.05571216344833374, 0.05168726667761803, 0.054652806371450424, 0.05950383096933365, 0.052205849438905716, 0.051408518105745316, 0.05771362781524658], [0.051101237535476685, 0.05265597254037857, 0.05074458569288254, 0.053684528917074203, 0.04988954961299896, 0.06589760631322861, 0.06234947219491005, 0.06536716222763062, 0.06039320304989815, 0.05375127121806145, 0.056936267763376236, 0.05184692144393921, 0.05273923650383949, 0.055836863815784454, 0.05473874881863594, 0.05352823808789253, 0.05362898111343384, 0.05491016432642937]], 'text_sha256': ['0fd5a83680ca586a4229b497bf30bf30d63ed7fdde9d6b40e98b876b9d6e73a2', '47bcac54396798c37e24deb37d60748b938c7105fbe930d29cc5edb75366f7f2', 'e136bd9f28d613014f0b21341d38c4d49351ba6117d92697e3603889c0495452'], 'source_script_version_id': 360593727, 'source_public_score': 0.606}

"""Exact memoization of three public examples from the scored source notebook.

The builder embeds _NTL_PUBLIC_CACHE with IDs, SHA-256 of the full input text,
class order and source probabilities. Every other input uses the full model path.
This branch checks integration and output format; it does not validate GPU model
execution or hidden-test accuracy. No hidden-test predictions are cached.
"""
import hashlib as _cache_hashlib
import json as _cache_json
import os as _cache_os
import time as _cache_time
import numpy as np
import pandas as pd


def _ntl_public_cache_order(frame, cache):
    if len(frame) != len(cache['ids']) or not {'id', 'text'}.issubset(frame.columns):
        return None
    ids = frame['id'].astype(str).tolist()
    if len(set(ids)) != len(ids) or set(ids) != set(cache['ids']):
        return None
    order = [cache['ids'].index(value) for value in ids]
    texts = frame['text'].fillna('').astype(str).tolist()
    for index, text in zip(order, texts):
        if _cache_hashlib.sha256(text.encode('utf-8')).hexdigest() != cache['text_sha256'][index]:
            return None
    return order


def _ntl_cache_softmax(values):
    exponent = np.exp(values - values.max(1, keepdims=True))
    return exponent / exponent.sum(1, keepdims=True)


def _ntl_initialize_public_cache(frame=None):
    if frame is None:
        candidates = ['/kaggle/input/competitions/name-that-llm', '/kaggle/input/name-that-llm']
        directory = next(path for path in candidates if _cache_os.path.isfile(path + '/test.csv'))
        frame = pd.read_csv(directory + '/test.csv')
    order = _ntl_public_cache_order(frame, _NTL_PUBLIC_CACHE)
    if order is None:
        print('PUBLIC CACHE MISS: running the complete Qwen/Gaussian/ATLAS inference path.', flush=True)
        return {'_NTL_RUN_FULL_MODELS': True, '_NTL_PUBLIC_CACHE_HIT': False}
    labels = list(_NTL_PUBLIC_CACHE['labels'])
    original = np.asarray(_NTL_PUBLIC_CACHE['original_probabilities'], dtype=np.float64)[order]
    atlas = np.asarray(_NTL_PUBLIC_CACHE['atlas_probabilities'], dtype=np.float64)[order]
    for probability in (original, atlas):
        if (probability.shape != (len(frame), len(labels)) or not np.isfinite(probability).all()
                or (probability < 0).any() or not np.allclose(probability.sum(1), 1, atol=1e-6)):
            raise ValueError('Public source cache has invalid probability values')
    frame = frame.copy()
    frame['id'] = frame['id'].astype(str)
    frame['text'] = frame['text'].fillna('').astype(str)
    groups = np.arange(1, len(frame) + 1, dtype=np.int64)
    print('PUBLIC CACHE HIT: exact ID and full-text hashes match the 3 upstream public examples; '
          'cached source probabilities, not a GPU inference test.', flush=True)
    return {'_NTL_RUN_FULL_MODELS': False, '_NTL_PUBLIC_CACHE_HIT': True,
            'test': frame, 'LABELS': labels, 'K': len(labels), 'INPUT_DIR': '/kaggle/input',
            'LOGIT': np.log(np.clip(original, 1e-12, 1)), 'ATLAS_PROB': atlas,
            'groups': groups, '_consensus_glm_groups': groups.copy(),
            '_consensus_glm_ids': frame['id'].tolist(), 'T0': _cache_time.time(),
            'softmax': _ntl_cache_softmax, 'time': _cache_time, 'json': _cache_json}


In [ ]:
from soft_constraints import soft_constraint_decode
_EXPERIMENT = {'key': 'hybrid_mrf', 'slug': 'ntl-exp1-cached-hybrid-soft-mrf', 'title': 'NTL Exp1 Cached Hybrid Soft MRF', 'file': 'NTL_Exp1_Hybrid_Soft_MRF.ipynb', 'members': ['svc', 'ts_q3b8', 'ts_q3b14', 'ts_llama8', 'ts_q25_7b'], 'baseline_weight': 0.75, 'svc_weight': 0.05, 'dhruv_weight': 0.2, 'atlas_mode': 'hybrid-selective', 'penalty_strength': 0.4, 'calibration': False, 'compared_with': 'cached-hybrid-v2; uncached hybrid v3 scored 0.602', 'change': 'Replace only hard Hungarian with guarded Soft MRF; same hybrid mixtures and ATLAS routing.'}
"""Notebook entry point: one baseline, distinct new members, one ATLAS and one decoder."""
import json,subprocess,sys,time
from pathlib import Path
import numpy as np
import pandas as pd
from hybrid_math import (validate_probabilities,align_probabilities,mix_candidates,softmax,
                         sinkhorn,selective_atlas_rows,dense_group_hungarian)

WORK=Path('/kaggle/working')
INPUT=Path('/kaggle/input')
DATA=next(path for path in [INPUT/'competitions/name-that-llm',INPUT/'name-that-llm'] if (path/'test.csv').exists())
train=pd.read_csv(DATA/'train.csv');test=pd.read_csv(DATA/'test.csv')
for frame in [train,test]:frame['text']=frame.text.fillna('').astype(str)
train.to_csv(WORK/'hybrid_train.csv',index=False)
test.to_csv(WORK/'hybrid_test.csv',index=False)
DEADLINE=_ntl_started+8.6*3600
stage_reports={}


def run_stage(name,command,seconds):
    if seconds<30:
        stage_reports[name]={'status':'SKIPPED_BUDGET','seconds':0}
        return False
    print('START',name,'budget seconds',round(seconds),flush=True)
    start=time.time()
    path=WORK/(name+'.log')
    with path.open('w',encoding='utf-8') as log:
        try:
            result=subprocess.run(command,cwd=WORK,stdout=log,stderr=subprocess.STDOUT,
                                  timeout=max(1,seconds),check=False)
            status='COMPLETE' if result.returncode==0 else 'ERROR'
        except subprocess.TimeoutExpired:status='TIME_BUDGET'
    stage_reports[name]={'status':status,'seconds':time.time()-start,'log':path.name}
    tail=path.read_text(encoding='utf-8',errors='replace')[-2500:]
    print('END',name,status,'seconds',round(time.time()-start),'\n'+tail,flush=True)
    (WORK/'stage-report.json').write_text(json.dumps(stage_reports,indent=2))
    return status=='COMPLETE'


cache=_ntl_initialize_public_cache(test)
cache_hit=cache['_NTL_PUBLIC_CACHE_HIT']
if cache_hit:
    labels=cache['LABELS']
    baseline=softmax(cache['LOGIT'])
    groups=cache['groups'];glm_groups=cache['_consensus_glm_groups']
    cached_atlas=cache['ATLAS_PROB']
else:
    if not run_stage('baseline',[sys.executable,'baseline_runtime.py'],min(7*3600,DEADLINE-time.time()-1800)):
        raise RuntimeError('Required baseline inference failed; inspect baseline.log')
    with np.load(WORK/'baseline-pre-atlas.npz',allow_pickle=False) as data:
        if data['ids'].tolist()!=test.id.astype(str).tolist():raise ValueError('Baseline row order differs')
        labels=data['labels'].tolist();baseline=data['probabilities']
        groups=data['groups'];glm_groups=data['glm_groups']
validate_probabilities(baseline,len(test),18)
np.savez_compressed(WORK/'hybrid_groups.npz',groups=groups,glm_groups=glm_groups,ids=test.id.astype(str).to_numpy(dtype=str))
request={'train_path':str(WORK/'hybrid_train.csv'),'test_path':str(WORK/'hybrid_test.csv'),
         'group_path':str(WORK/'hybrid_groups.npz'),'atlas_rows_path':str(WORK/'atlas_rows.npy')}
request_path=WORK/'hybrid-request.json'
request_path.write_text(json.dumps(request))
member_outputs={}
member_reports={}
# Full-data adapters only, no five-fold expansion. Candidates are isolated from one another.
for name in _EXPERIMENT['members']:
    remaining=DEADLINE-time.time()
    reserve=1800 if not cache_hit else 180
    budget=min(1800 if name=='svc' else 2700,remaining-reserve)
    output=WORK/(name+'.npz')
    command=[sys.executable,'candidate_runtime.py',name,'--request',str(request_path),'--output',str(output)]
    # Cached Gaussian needs no extra train forward pass; preserve it when possible.
    complete=run_stage(name,command,budget)
    if not complete and name!='svc' and Path(str(output)+'.plain.npz').exists():
        output=Path(str(output)+'.plain.npz')
        stage_reports[name]['using_pre_gaussian_checkpoint']=True
    elif not complete:
        continue
    with np.load(output,allow_pickle=False) as data:
        if data['ids'].tolist()!=test.id.astype(str).tolist():raise ValueError(name+' row order differs')
        member_outputs[name]=align_probabilities(data['probabilities'],data['labels'].tolist(),labels)
    member_reports[name]=json.loads(Path(str(output)+'.json').read_text())
if cache_hit and set(member_outputs) != set(_EXPERIMENT['members']):
    raise RuntimeError('Every member requested by this ablation must complete; inspect stage logs')
dhruv_weights={'ts_q3b8':0.30,'ts_q3b14':0.40,'ts_llama8':0.20,'ts_q25_7b':0.10}
available={name:weight for name,weight in dhruv_weights.items() if name in member_outputs}
candidates={}
if available:
    candidates['dhruv']=sum(member_outputs[name]*weight for name,weight in available.items())/sum(available.values())
if 'svc' in member_outputs:candidates['vishak_svc']=member_outputs['svc']
pre_atlas,used_weights=mix_candidates(baseline,candidates,{'dhruv':0.20,'vishak_svc':0.05})
if cache_hit:
    atlas_rows=np.arange(len(test));atlas=cached_atlas
    atlas_report={'source':'exact_public_example_cache','requested':len(test),'completed':len(test)}
else:
    wanted=selective_atlas_rows(pre_atlas,margin=0.30)
    np.save(WORK/'atlas_rows.npy',wanted)
    atlas_rows=np.array([],dtype=int);atlas=np.empty((0,len(labels)))
    atlas_report={'requested':len(wanted),'completed':0}
    if len(wanted):
        budget=max(0,DEADLINE-time.time()-180)
        output=WORK/'atlas-partial.npz'
        run_stage('atlas',[sys.executable,'atlas_runtime.py','--request',str(request_path),
                   '--output',str(output),'--seconds',str(max(0,budget-30))],budget)
        if output.exists():
            with np.load(output,allow_pickle=False) as data:
                atlas_rows=data['rows']
                if (len(set(atlas_rows.tolist()))!=len(atlas_rows) or not set(atlas_rows).issubset(set(wanted))
                        or data['ids'].tolist()!=test.id.astype(str).iloc[atlas_rows].tolist()):
                    raise ValueError('ATLAS row IDs or selected rows differ')
                atlas=align_probabilities(data['probabilities'],data['labels'].tolist(),labels)
        atlas_report['completed']=len(atlas_rows)
    atlas_report['source']='new_gpu_inference'
mixed=pre_atlas.copy()
if len(atlas_rows):mixed[atlas_rows]=0.90*pre_atlas[atlas_rows]+0.10*atlas
probability=sinkhorn(mixed)
predictions,assignment_report=soft_constraint_decode(probability,groups,glm_groups,
    {'penalty_strength':_EXPERIMENT['penalty_strength']})
assignment_report['method']='guarded-soft-mrf'
pd.DataFrame(assignment_report['block_decisions']).to_csv(WORK/'mrf-groups.csv',index=False)
_changed=np.flatnonzero(predictions!=probability.argmax(1))
pd.DataFrame({'id':test.id.astype(str).iloc[_changed].to_numpy(),
    'argmax_model':np.asarray(labels)[probability.argmax(1)[_changed]],
    'mrf_model':np.asarray(labels)[predictions[_changed]]}).to_csv(WORK/'mrf-changes.csv',index=False)
submission=pd.DataFrame({'id':test.id.astype(str),'model':np.asarray(labels)[predictions]})
if (submission.empty or submission.isna().any().any() or not submission.id.is_unique
        or submission.id.tolist()!=test.id.astype(str).tolist() or not submission.model.isin(labels).all()):
    raise ValueError('Submission columns, IDs or labels are invalid')
submission.to_parquet(WORK/'submission.parquet',index=False)
if not pd.read_parquet(WORK/'submission.parquet').equals(submission):raise ValueError('Parquet round-trip differs')
pd.DataFrame({'id':test.id.astype(str),'model':np.asarray(labels)[baseline.argmax(1)]}).to_parquet(WORK/'baseline-pre-atlas.parquet',index=False)
np.savez_compressed(WORK/'hybrid-probabilities.npz',baseline_pre_atlas=baseline,hybrid_pre_atlas=pre_atlas,
    probabilities=probability,labels=np.asarray(labels),ids=test.id.astype(str).to_numpy(dtype=str),
    selected_predictions=predictions,groups=groups,glm_groups=glm_groups,
    **{name:values for name,values in member_outputs.items()})
report={'pipeline':'ntl-ablation-hybrid_mrf-v1','rows':len(test),
    'baseline_source_script_version_id':360593727,'baseline_source_public_score':0.606,
    'experiment_public_score':None,'baseline_public_cache_hit':cache_hit,
    'baseline_full_model_inference_executed':not cache_hit,'new_members_executed':list(member_outputs),
    'train_feature_cache_used':True,'qwen_lora_training':False,'adapter_selection':'full-data-only','elapsed_seconds':time.time()-_ntl_started,
    'weights':used_weights,'dhruv_member_weights_used':available,'member_reports':member_reports,
    'stages':stage_reports,'atlas':atlas_report,'postprocessing':assignment_report,
    'soft_mrf_enabled':True,'experiment':_EXPERIMENT,'submission_validated':True,
    'llama_base_source':'unsloth/Meta-Llama-3.1-8B@e9a141a2091ea561b96483212645a2a05e6f99fc; staged as verified Kaggle input',
    'example_run_does_not_validate_hidden_accuracy':True}
if _EXPERIMENT['calibration']:
    _cal=json.loads((WORK/'svc-calibration-report.json').read_text())
    if _cal['known_prompt_overlap']!=0 or len(_cal['folds'])!=3:raise ValueError('Invalid SVC calibration')
    report['svc_calibration']=_cal
(WORK/'hybrid-report.json').write_text(json.dumps(report,indent=2))
print('HYBRID COMPLETE',json.dumps({key:report[key] for key in ['rows','weights','new_members_executed','elapsed_seconds','baseline_public_cache_hit']}),flush=True)
